<a href="https://colab.research.google.com/github/ericyoc/dissertation_code_and_data/blob/main/dissertation_code_and_data_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#import os
#os.remove('/content/drive/My Drive/clean_models/clean_model_TinyImageNet.pth')
#print("Deleted old model")

In [ ]:
import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

# SET GLOBAL RANDOM SEEDS FOR REPRODUCIBILITY
import random
import numpy as np
import torch

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)  # for multi-GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seeds set to {RANDOM_SEED} for reproducibility")

!pip install -q torchattacks
import torchattacks

#global defense_type

global randomization_defense
randomization_defense = "combined_randomization"  # Options: "random_resizing", "random_cropping", "random_rotation", "combined_randomization"

global input_transformation
input_transformation = "combined_input_transformation"  # Options: "image_quilting", "jpeg_compression", "bit_depth_reduction", "gaussian_noise_defense", "combined_input_transformation"

global defense_type
defense_type = "input_transformation"  # Options: "adversarial_training", "randomization", "input_transformation"

#Pick a compounded adversarial attack from the following: fgsm_cw_attack, fgsm_pgd_attack, cw_pgd_attack
global compounded_attack_name

compounded_attack_name = "fgsm_pgd_attack"
#compounded_attack_name = "fgsm_cw_attack"
#compounded_attack_name = "cw_pgd_attack"

# Set the dataset_name
global dataset_name
dataset_name = 'TinyImageNet'  # 'MNIST' or 'EMNIST' or 'SVHN' or 'USPS' or 'Semeion' or 'TinyImageNet' or 'TrafficSigns' or 'ReflectSign'

# Global variable to store TinyImageNet selected classes for label remapping
tinyimagenet_selected_classes = None

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Access the mounted drive
drive_path = '/content/drive/My Drive/'

import torch
global device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

global num_epochs
#num_epochs = 10 # MNIST and EMNIST "Digits"
#num_epochs = 60 # Traffic Signs
#num_epochs = 100 # ReflectSign
num_epochs = 120  # TinyImageNet

!pip install -q cirq
import cirq
!pip install -q cirq-google
import cirq_google

import sys
import pkg_resources
import importlib

def get_package_versions(packages):
    versions = {}
    for package in packages:
        try:
            module = importlib.import_module(package)
            if hasattr(module, '__version__'):
                versions[package] = module.__version__
            elif package == 'cirq':
                versions[package] = cirq.__version__
            else:
                versions[package] = 'Not Found'
        except ImportError:
            versions[package] = 'Not Installed'
    return versions

# Specify the list of packages you want to check
packages_to_check = ["torch", "torchvision", "torchattacks", "torchvision", "numpy", "tabulate", "cirq","cirq_google"]

# Call the function to get package versions
versions = get_package_versions(packages_to_check)

# Get the Python version
python_version = sys.version.split()[0]

# Call the function to get package versions
versions = get_package_versions(packages_to_check)

# Print the Python version
print(f"Python version: {python_version}")

# Print the package versions
for package_name, version in versions.items():
    print(f"{package_name}: {version}")

import multiprocessing as mp
try:
    mp.set_start_method("spawn")
except RuntimeError:
    pass  # Already set

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

"""theta is the rotation angle applied to even-indexed qubits (0, 2, 4, ...) using the cirq.ry(theta) gate.

phi is the rotation angle applied to odd-indexed qubits (1, 3, 5, ...) using the cirq.ry(phi) gate.

These rotation gates introduce controlled dynamics into the quantum circuit, and by varying the values of theta and phi, we can modify the evolution of the quantum state and potentially increase the expressive power of the circuit.

During training, theta and phi are initialized as PyTorch tensors with requires_grad=True, allowing their values to be optimized via backpropagation
"""

# Initialize trainable parameters for the quantum circuit
global theta
global phi
theta = torch.randn(1, requires_grad=True, device=device)
phi = torch.randn(1, requires_grad=True, device=device)

"""The create_quantum_circuit function has two types of quantum gates being used: rotation gates and entangling gates.

Rotation Gates:
The code uses the cirq.ry gate, which represents a rotation around the Y-axis of the Bloch sphere.
The cirq.ry(theta)(qubit) applies a rotation by an angle theta around the Y-axis to the specified qubit.
In the code, there are two fixed rotation angles used: theta and phi.
The rotation gates are applied to each qubit in an alternating manner based on the qubit index.
If the qubit index is even (i % 2 == 0), a rotation by angle theta is applied using cirq.ry(theta)(qubit).
If the qubit index is odd (i % 2 != 0), a rotation by angle phi is applied using cirq.ry(phi)(qubit).
Rotation gates introduce single-qubit operations that can manipulate the state of individual qubits.

Entangling Gates:
The code uses the cirq.CNOT gate, which represents the controlled-NOT operation.
The cirq.CNOT(control_qubit, target_qubit) applies a CNOT gate with control_qubit as the control and target_qubit as the target.
In the code, CNOT gates are applied between pairs of adjacent qubits.
The loop for i in range(num_qubits - 1) iterates over the qubits, skipping the last one.

For each pair of adjacent qubits qubits[i] and qubits[i+1], a CNOT gate is appended to the circuit using circuit.append(cirq.CNOT(qubits[i], qubits[i+1])).
CNOT gates create entanglement between qubits, allowing for multi-qubit operations and introducing correlations between the states of different qubits.
The purpose of using these quantum gates in the circuit is to introduce quantum operations that can manipulate the state of the qubits. The rotation gates (cirq.ry) allow for single-qubit transformations, while the entangling gates (cirq.CNOT) create entanglement and enable multi-qubit operations.

By applying a combination of rotation gates and entangling gates, the quantum circuit can perform quantum computations and potentially capture complex patterns and correlations in the data. The specific choice of gates and their arrangement depends on the desired quantum algorithm and the problem being solved.

Note: The code uses fixed rotation angles (theta and phi) for simplicity, but in practice, these angles can be parameterized and learnable during the training process of the hybrid model.
"""

import math

def create_quantum_circuit(theta, phi, output_dim):

    # Check if output_dim is valid
    if output_dim <= 0:
        print(f"Output dimension {output_dim} is invalid. Returning an empty circuit.")
        return cirq.Circuit()

    # Number of qubits required for output_dim classes
    num_qubits = int(math.ceil(math.log2(output_dim)))

    # Create qubits
    qubits = cirq.LineQubit.range(num_qubits)

    # Initialize quantum circuit
    circuit = cirq.Circuit()

    # Apply fixed rotations to each qubit
    theta = 0.5
    phi = 0.3
    for i, qubit in enumerate(qubits):
        if i % 2 == 0:
            circuit.append(cirq.ry(theta)(qubit))
        else:
            circuit.append(cirq.ry(phi)(qubit))

    # Apply entangling gates (e.g., CNOT gates)
    for i in range(num_qubits - 1):
        circuit.append(cirq.CNOT(qubits[i], qubits[i+1]))

    return circuit

def simulate_circuit(circuit, device):

    if device == "gpu":
        # Create a GPU-accelerated simulator
        simulator = cirq.TensorflowSimulator(tensor_env=tf.distribute.cluster_resolver.TPUClusterResolver())
    else:
        # Create a CPU simulator
        simulator = cirq.Simulator()

    result = simulator.simulate(circuit)
    final_state_vector = result.final_state_vector

    return final_state_vector

# Create the quantum circuit
quantum_circuit = create_quantum_circuit(theta, phi, 10)

# Print the circuit
print(quantum_circuit)

# Simulate the circuit on CPU
cpu_state_vector = simulate_circuit(quantum_circuit, device="cpu")
print("CPU state vector:", cpu_state_vector)

# Simulate the circuit on GPU (if available)
try:
    gpu_state_vector = simulate_circuit(quantum_circuit, device="gpu")
    print("GPU state vector:", gpu_state_vector)
except Exception as e:
    print("GPU acceleration is not available on this system.")
    print(e)

# Set correct output_dim based on the dataset manually
if dataset_name == 'ReflectSign':
    output_dim = 55
elif dataset_name == 'TrafficSigns':
    output_dim = 4
else:
    output_dim = 10


def hybrid_forward(input_data, classical_model, theta, phi, device, output_dim):
    # Move input data to the specified device
    input_data = input_data.to(device)

    # Pass input through classical model
    classical_output = classical_model(input_data)

    # Construct quantum circuit based on classical output
    quantum_circuit = create_quantum_circuit(theta, phi, output_dim)

    # Simulate quantum circuit and extract results
    simulator = cirq.Simulator()
    result = simulator.simulate(quantum_circuit)
    quantum_output_amplitudes = result.final_state_vector

    # Get the batch size from the classical output
    batch_size = classical_output.size(0)

    # Reshape classical output to match quantum output dimensions
    classical_output = classical_output.view(batch_size, -1)

    # Compute the squared amplitudes (probabilities) of the quantum output
    num_qubits = int(math.ceil(math.log2(output_dim)))
    expected_quantum_output_size = 2 ** num_qubits
    quantum_output_probabilities = np.square(np.abs(quantum_output_amplitudes))
    quantum_output_probabilities = quantum_output_probabilities.reshape(expected_quantum_output_size)[:output_dim]

    # Repeat the quantum output probabilities for each batch element
    quantum_output_probabilities = np.tile(quantum_output_probabilities, (batch_size, 1))

    # Convert quantum output probabilities to PyTorch tensor
    quantum_output_probabilities = torch.from_numpy(quantum_output_probabilities).to(device)

    # Combine classical and quantum outputs using a linear combination
    alpha = 0.5  # Weight for the classical output
    beta = 0.5  # Weight for the quantum output
    hybrid_output = alpha * classical_output + beta * quantum_output_probabilities

    # Apply log_softmax to the hybrid output
    hybrid_output = F.log_softmax(hybrid_output, dim=1)

    return hybrid_output

"""CNN architecture is designed to recognize digits 0 to 9 in the MNIST type dataset.

Input:
The MNIST dataset consists of grayscale images of handwritten digits.
Each image has a size of 28x28 pixels.
The input to the CNN is expected to be a tensor of shape (batch_size, 1, 28, 28), where 1 represents the single color channel (grayscale).

Convolutional Layers:
The first convolutional layer (self.conv1) has 1 input channel (grayscale) and 4 output feature maps. It uses a kernel size of 5x5 to learn local patterns in the input image.

The second convolutional layer (self.conv2) has 4 input feature maps (output from conv1) and 10 output feature maps. It also uses a kernel size of 5x5 to learn higher-level features.

Max pooling is applied after each convolutional layer to downsample the feature maps and reduce spatial dimensions.

Fully Connected Layers:
After the convolutional layers, the feature maps are flattened into a 1D vector.
The flattened vector is passed through two fully connected layers (self.fc1 and self.fc2) to learn non-linear combinations of the extracted features.
The output of the last fully connected layer (self.fc2) has 10 units, corresponding to the 10 digit classes (0 to 9).

Output:
The output of the CNN is a tensor of shape (batch_size, 10), representing the predicted probabilities for each digit class.
The log_softmax activation function is applied to the output to obtain normalized log probabilities.

During training, the CNN learns to map the input images to their corresponding digit labels by adjusting the weights of the convolutional and fully connected layers. The network learns to extract relevant features and patterns from the images that are discriminative for digit recognition.

At inference time, given an input image of a handwritten digit, the trained CNN will predict the probability distribution over the 10 digit classes. The class with the highest probability is typically considered as the predicted digit.
"""

# Define model
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        if dataset_name == 'TinyImageNet':
            # SIMPLIFIED ARCHITECTURE FOR TINYIMAGENET (matching TrafficSigns success)
            # TrafficSigns proved that SHALLOW networks work better for small filtered datasets
            # 3 convolutional layers (NOT 5) - prevents overfitting
            # WIDENED channels (32->64->128) for higher capacity with more training data

            # First convolutional layer
            self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
            self.bn1 = nn.BatchNorm2d(32)

            # Second convolutional layer
            self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
            self.bn2 = nn.BatchNorm2d(64)

            # Third convolutional layer
            self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
            self.bn3 = nn.BatchNorm2d(128)

            # After 3 max pools on 64x64: 64->32->16->8
            # Flatten: 128 * 8 * 8 = 8192
            self.fc1 = nn.Linear(128 * 8 * 8, 256)
            self.dropout = nn.Dropout(p=0.6)  # Increased from 0.5 to reduce overfitting

            # Output layer
            self.fc2 = nn.Linear(256, 5)

            self.is_tiny = True
            self.is_tinyimagenet_deep = False  # Changed to False - using shallow architecture

        elif dataset_name == 'TrafficSigns' or dataset_name == 'ReflectSign':
            # SHALLOW ARCHITECTURE FOR TRAFFICSIGNS/REFLECTSIGN
            # First convolutional layer
            # Input: 3 channels (RGB image)
            # Output: 16 feature maps
            # Kernel size: 3x3
            self.conv1 = nn.Conv2d(3, 16, 3, padding=1)

            # TrafficSigns-specific: Add Batch Normalization after conv1
            if dataset_name == 'TrafficSigns':
                self.bn1 = nn.BatchNorm2d(16)

            # Second convolutional layer
            # Input: 16 feature maps
            # Output: 32 feature maps
            self.conv2 = nn.Conv2d(16, 32, 3, padding=1)

            # TrafficSigns-specific: Add Batch Normalization after conv2
            if dataset_name == 'TrafficSigns':
                self.bn2 = nn.BatchNorm2d(32)

            # Third convolutional layer
            # Input: 32 feature maps
            # Output: 64 feature maps
            # Kernel size: 3x3
            self.conv3 = nn.Conv2d(32, 64, 3, padding=1)

            # TrafficSigns-specific: Add Batch Normalization after conv3
            if dataset_name == 'TrafficSigns':
                self.bn3 = nn.BatchNorm2d(64)

            # First fully connected layer
            # Input: 64 * 8 * 8 features (after 3 poolings on 64x64)
            # Output: 128 features
            self.fc1 = nn.Linear(64 * 8 * 8, 128)

            # TrafficSigns-specific: Add Dropout before fc2 for regularization
            if dataset_name == 'TrafficSigns':
                self.dropout = nn.Dropout(p=0.5)

            # Second fully connected layer
            # Input: 128 features
            # Output: num_classes (dataset-specific)
            if dataset_name == 'ReflectSign':
                self.fc2 = nn.Linear(128, 55)  # ReflectSign has 55 classes
            elif dataset_name == 'TrafficSigns':
                self.fc2 = nn.Linear(128, 4)   # TrafficSigns has 4 classes
            else:
                self.fc2 = nn.Linear(128, 10)  # TinyImageNet and others have 10 classes

            self.is_tiny = True
            self.is_tinyimagenet_deep = False

        else:
            # First convolutional layer
            # Input: 1 channel (grayscale image)
            # Output: 4 feature maps
            # Kernel size: 5x5
            self.conv1 = nn.Conv2d(1, 4, 5)

            # Second convolutional layer
            # Input: 4 feature maps
            # Output: 10 feature maps
            # Kernel size: 5x5
            self.conv2 = nn.Conv2d(4, 10, 5)

            # First fully connected layer
            # Input: 160 features (flattened output from conv2)
            # Output: 80 features
            self.fc1 = nn.Linear(160, 80)

            # Second fully connected layer
            # Input: 80 features
            # Output: 10 features (corresponding to 10 classes)
            self.fc2 = nn.Linear(80, 10)

            self.is_tiny = False

    def forward(self, x):
        if self.is_tiny:
            if hasattr(self, 'is_tinyimagenet_deep') and self.is_tinyimagenet_deep:
                # DEEPER TINYIMAGENET ARCHITECTURE
                x = F.relu(self.bn1(self.conv1(x)))
                x = F.max_pool2d(x, 2)  # 64→32

                x = F.relu(self.bn2(self.conv2(x)))
                x = F.max_pool2d(x, 2)  # 32→16

                x = F.relu(self.bn3(self.conv3(x)))
                x = F.max_pool2d(x, 2)  # 16→8

                x = F.relu(self.bn4(self.conv4(x)))
                x = F.max_pool2d(x, 2)  # 8→4

                x = F.relu(self.bn5(self.conv5(x)))
                x = F.max_pool2d(x, 2)  # 4→2

                x = x.view(-1, 512 * 2 * 2)  # Flatten

                x = F.relu(self.bn_fc(self.fc1(x)))
                x = self.dropout1(x)

                x = F.relu(self.fc2(x))
                x = self.dropout2(x)

                x = self.fc3(x)
            else:
                # SHALLOW ARCHITECTURE (TrafficSigns, ReflectSign, TinyImageNet)
                # Apply first convolutional layer and ReLU activation
                x = F.relu(self.conv1(x))

                # Apply Batch Normalization
                if dataset_name in ['TrafficSigns', 'TinyImageNet']:
                    x = self.bn1(x)

                # Apply max pooling with kernel size 2x2
                x = F.max_pool2d(x, 2)

                # Apply second convolutional layer and ReLU activation
                x = F.relu(self.conv2(x))

                # Apply Batch Normalization
                if dataset_name in ['TrafficSigns', 'TinyImageNet']:
                    x = self.bn2(x)

                # Apply max pooling with kernel size 2x2
                x = F.max_pool2d(x, 2)

                # Apply third convolutional layer and ReLU activation
                x = F.relu(self.conv3(x))

                # Apply Batch Normalization
                if dataset_name in ['TrafficSigns', 'TinyImageNet']:
                    x = self.bn3(x)

                # Apply max pooling with kernel size 2x2
                x = F.max_pool2d(x, 2)

                # Flatten the output from the convolutional layers
                if dataset_name == 'TinyImageNet':
                    x = x.view(-1, 128 * 8 * 8)  # TinyImageNet: 128 channels
                else:
                    x = x.view(-1, 64 * 8 * 8)   # TrafficSigns: 64 channels

                # Apply first fully connected layer and ReLU activation
                x = F.relu(self.fc1(x))

                # Apply Dropout for regularization
                if dataset_name in ['TrafficSigns', 'TinyImageNet']:
                    x = self.dropout(x)

                # Apply second fully connected layer
                x = self.fc2(x)

        else:
            # Apply first convolutional layer and ReLU activation
            x = F.relu(self.conv1(x))

            # Apply max pooling with kernel size 2x2
            x = F.max_pool2d(x, 2)

            # Apply second convolutional layer and ReLU activation
            x = F.relu(self.conv2(x))

            # Apply max pooling with kernel size 2x2
            x = F.max_pool2d(x, 2)

            # Flatten the output from the convolutional layers
            x = x.view(-1, 160)

            # Apply first fully connected layer and ReLU activation
            x = F.relu(self.fc1(x))

            # Apply second fully connected layer
            x = self.fc2(x)

        # Apply log softmax activation to get class probabilities
        return F.log_softmax(x, dim=1)

classical_model = CNN().to(device)

print("Model Architecture:")
print(classical_model)

"""The Hybrid Neural Network (HNN) combines a classical model (e.g., CNN) with a quantum circuit. The purpose of the HNN is to enhance the performance of the classical model by leveraging the capabilities of quantum computing.

The HNN supports the CNN in recognizing MNIST digits 0-9 as explained below:

Integration with the Classical Model:
The HNN takes the classical model (CNN) as an input parameter (classical_model) during initialization.
The classical model is stored as an attribute of the HNN using self.classical_model = classical_model.
This allows the HNN to access and utilize the features and predictions of the CNN.

Quantum Circuit Parameters:
The HNN initializes trainable parameters (self.theta and self.phi) for the quantum circuit.
These parameters represent the angles of rotation gates in the quantum circuit and are learned during the training process.
The quantum circuit introduces additional non-linearity and expressiveness to the model, potentially enhancing its ability to capture complex patterns in the data.

Forward Pass:
In the forward method of the HNN, the input data (input_data) is passed to the hybrid_forward function along with the classical model, quantum circuit parameters, device, and output dimension.
The hybrid_forward function is responsible for integrating the classical model with the quantum circuit.
It takes the features extracted by the CNN and uses them as input to the quantum circuit.
The quantum circuit applies quantum operations based on the learned parameters (self.theta and self.phi) to transform the input features.
The output of the quantum circuit is then processed and combined with the classical model's predictions to produce the final output.

Hybrid Learning:
During the training process, both the classical model (CNN) and the quantum circuit parameters (self.theta and self.phi) are optimized jointly.
The gradients are propagated through the hybrid model, allowing the CNN and the quantum circuit to learn and adapt together.
The quantum circuit can potentially learn to extract additional features or patterns that complement the CNN's capabilities.

Output Dimension:
The HNN's output dimension (output_dim) is set to 10, corresponding to the 10 digit classes (0 to 9) in the MNIST dataset.
The hybrid model's final output represents the predicted probabilities for each digit class.

By combining the CNN with a quantum circuit, the HNN aims to leverage the strengths of both classical and quantum computing. The quantum circuit can potentially capture additional patterns and correlations in the data that the CNN alone might miss. The hybrid approach allows for the exploration of quantum-enhanced feature extraction and classification.
"""

class HNN(nn.Module):
    def __init__(self, classical_model, device, output_dim=10):
        super(HNN, self).__init__()

        # Store the classical model as an attribute
        self.classical_model = classical_model

        # Store the output dimension as an attribute
        self.output_dim = output_dim

        # Initialize trainable parameters for the quantum circuit
        # TinyImageNet needs more quantum parameters
        if dataset_name == 'TinyImageNet':
            self.theta = nn.Parameter(torch.randn(128))
            self.phi = nn.Parameter(torch.randn(128))
        else:
            self.theta = nn.Parameter(torch.randn(1))
            self.phi = nn.Parameter(torch.randn(1))

        # Store the device as an attribute
        self.device = device

        # Move the model to the specified device if it's not None
        if device is not None:
            self.to(device)

    def forward(self, input_data):
        # Call the hybrid_forward function with the input data, classical model,
        # quantum circuit parameters (theta and phi), device, and output dimension
        return hybrid_forward(input_data, self.classical_model, self.theta, self.phi, self.device, self.output_dim)

# Create an instance of the hybrid model
hybrid_model = HNN(classical_model, device=device, output_dim=10).to(device)

# Print the hybrid model architecture
print("Hybrid Model Architecture:")
print(hybrid_model)

# TrafficSigns-specific: Add weight decay for regularization
if dataset_name == 'TrafficSigns':
    optimizer = optim.Adam(classical_model.parameters(), lr=0.001, weight_decay=1e-4)
    print(f"TrafficSigns: Using Adam with lr=0.001, weight_decay=1e-4 for regularization")
# TinyImageNet-specific: Higher LR with scheduler, stronger weight decay
elif dataset_name == 'TinyImageNet':
    optimizer = optim.Adam(classical_model.parameters(), lr=0.001, weight_decay=3e-4, betas=(0.9, 0.999))
    print(f"TinyImageNet: Using Adam with lr=0.001 (with scheduler), weight_decay=3e-4 for regularization")
else:
    optimizer = optim.Adam(classical_model.parameters())

loss_fn = nn.NLLLoss()

import gzip
import numpy as np
from torchvision import transforms

def load_emnist_digits(data_dir, train=True, transform=None, filtered=range(10)):
    if train:
        image_file = 'emnist-digits-train-images-idx3-ubyte.gz'
        label_file = 'emnist-digits-train-labels-idx1-ubyte.gz'
    else:
        image_file = 'emnist-digits-test-images-idx3-ubyte.gz'
        label_file = 'emnist-digits-test-labels-idx1-ubyte.gz'

    image_path = os.path.join(data_dir, image_file)
    label_path = os.path.join(data_dir, label_file)

    with gzip.open(image_path, 'rb') as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(label_path, 'rb') as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)

    dataset = EMNISTDataset(images, labels, transform)

    if filtered is not None:
        digits_to_keep = list(filtered)  # Convert the range object to a list
        mask = np.isin(dataset.labels, digits_to_keep)
        dataset.images = dataset.images[mask]
        dataset.labels = dataset.labels[mask]

    return dataset

from torch.utils.data import Subset

def filtered_dataset_EMNIST(train_set, test_set, max_train_per_class=200, max_test_per_class=50):
    print("Sampling for multi-class classification (classes 0 to 9)...")
    print(f"Using FAST subset - {max_train_per_class} train and {max_test_per_class} test per class")
    print(f"Total: {max_train_per_class * 10} training, {max_test_per_class * 10} test images")

    train_labels = torch.tensor(train_set.targets)
    test_labels = torch.tensor(test_set.targets)

    train_indices = []
    test_indices = []

    for cls in range(10):
        print(f"Filtering class {cls}...")

        cls_train = (train_labels == cls).nonzero(as_tuple=True)[0][:max_train_per_class]
        cls_test = (test_labels == cls).nonzero(as_tuple=True)[0][:max_test_per_class]

        train_indices.extend(cls_train.tolist())
        test_indices.extend(cls_test.tolist())

    return Subset(train_set, train_indices), Subset(test_set, test_indices)

import gzip
import numpy as np
import torch
from torch.utils.data import Dataset
from torchvision.transforms import ToTensor

class EMNISTDigits(Dataset):
    def __init__(self, root, train=True, transform=None, target_transform=None):
        self.root = root
        self.train = train
        self.transform = transform
        self.target_transform = target_transform

        prefix = 'train' if self.train else 'test'
        image_file = f'{self.root}/emnist-digits-{prefix}-images-idx3-ubyte.gz'
        label_file = f'{self.root}/emnist-digits-{prefix}-labels-idx1-ubyte.gz'

        with gzip.open(image_file, 'rb') as f:
            self.images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)

        with gzip.open(label_file, 'rb') as f:
            self.labels = np.frombuffer(f.read(), np.uint8, offset=8)

    def __getitem__(self, index):
        image = self.images[index]
        label = self.labels[index]

        if self.transform is not None:
            image = self.transform(image)

        if self.target_transform is not None:
            label = self.target_transform(label)

        return image, label

    def __len__(self):
        return len(self.images)

!ls "/content/drive/My Drive/data/EMNIST/raw/"

from torch.utils.data import Dataset
from PIL import Image
import os

class TrafficSignsDataset(Dataset):
    def __init__(self, image_dir, label_map, label_to_idx, transform=None):
        self.image_dir = image_dir
        self.label_map = label_map
        self.label_to_idx = label_to_idx  # store this for label_map reconstruction
        self.transform = transform
        self.filenames = [f for f in os.listdir(image_dir) if f in label_map]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_name = self.filenames[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        label = self.label_to_idx[self.label_map[img_name]]
        if self.transform:
            image = self.transform(image)
        return image, label

import xml.etree.ElementTree as ET

def build_traffic_sign_label_map(annotation_dir):
    label_map = {}
    for fname in os.listdir(annotation_dir):
        if not fname.endswith('.xml'):
            continue
        try:
            root = ET.parse(os.path.join(annotation_dir, fname)).getroot()
            filename = root.find('filename').text
            obj = root.find('object')
            label = obj.find('name').text if obj is not None else 'unknown'
            label_map[filename] = label
        except Exception as e:
            print(f"XML error in {fname}: {e}")
    return label_map

def save_traffic_sign_wordlist(label_map, output_file):
    classes = sorted(set(label_map.values()))
    with open(output_file, 'w') as f:
        for label in classes:
            f.write(f"{label}\n")
    return classes

# Global class index-to-label mapping
idx_to_class = None

def prepare_reflectsign_dataset(base_dir="/content/drive/MyDrive/datasets/SignRetroreflectivity"):
    import pandas as pd
    import os

    tabular_path = os.path.join(base_dir, "Tabular Data", "Tabular Data.xlsx")
    wordlist_path = os.path.join(base_dir, "wordlist.txt")

    # === Load tabular data ===
    tabular_df = pd.read_excel(tabular_path, sheet_name='Tabular Data')

    # === Create wordlist.txt ===
    wordlist_entries = []
    for i in range(1, 189):
        filename = f"{i}.png"
        try:
            label = str(tabular_df.iloc[i - 1]["Comment"]).strip()
        except Exception:
            label = "Unknown"
        wordlist_entries.append((filename, label))

    with open(wordlist_path, "w") as f:
        for filename, label in wordlist_entries:
            f.write(f"{filename} {label}\n")

    print(f" wordlist.txt saved to: {wordlist_path}")

    # === Merge metadata ===
    wordlist_df = pd.DataFrame(wordlist_entries, columns=["filename", "label"])
    tabular_subset = tabular_df.iloc[:188].copy().reset_index(drop=True)
    tabular_subset["filename"] = [f"{i+1}.png" for i in range(188)]
    final_df = pd.merge(wordlist_df, tabular_subset, on="filename", how="left")

    # Save merged metadata to CSV
    csv_path = os.path.join(base_dir, "retroreflectivity_image_metadata.csv")
    final_df.to_csv(csv_path, index=False)
    print(f" Merged CSV saved to: {csv_path}")

    return final_df

import matplotlib.pyplot as plt

def display_images_from_wordlist(base_dir="/content/drive/MyDrive/datasets/SignRetroreflectivity"):
    wordlist_path = os.path.join(base_dir, "wordlist.txt")
    train_dir = os.path.join(base_dir, "Train Img")
    test_dir = os.path.join(base_dir, "Test Img")

    with open(wordlist_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue
        try:
            filename, label = line.split(" ", 1)

            # Try Train folder first, then Test
            img_path_train = os.path.join(train_dir, filename)
            img_path_test = os.path.join(test_dir, filename)

            if os.path.exists(img_path_train):
                img_path = img_path_train
            elif os.path.exists(img_path_test):
                img_path = img_path_test
            else:
                raise FileNotFoundError(f"Not found in Train or Test: {filename}")

            img = Image.open(img_path)
            print(f"{filename} → Label: {label}")
            plt.imshow(img)
            plt.title(f"{filename} | Label: {label}")
            plt.axis('off')
            plt.show()

        except Exception as e:
            print(f"⚠️ Could not open {filename}: {e}")

if dataset_name == 'ReflectSign':
   display_images_from_wordlist()

"""The MNIST (Modified National Institute of Standards and Technology) dataset is a large collection of handwritten digits commonly used for training and testing machine learning models, particularly in the field of computer vision and image classification.

The MNIST dataset contains a total of 70,000 images of handwritten digits. These images are divided into two subsets:

Training set: The training set consists of 60,000 images. This set is used to train the machine learning models, allowing them to learn patterns and features from the labeled examples.
Test set: The test set contains the remaining 10,000 images. This set is used to evaluate the performance and accuracy of the trained models on unseen data.
Each image in the MNIST dataset is a grayscale image with a size of 28x28 pixels. The digits in the images are centered and normalized to fit within the 28x28 pixel frame.

The EMNIST (Extended MNIST) dataset is an extension of the original MNIST dataset, providing additional handwritten character classes beyond just digits. The EMNIST dataset consists of several subsets, including EMNIST Balanced, EMNIST Letters, EMNIST Digits, EMNIST MNIST, and EMNIST By_Merge.

Specifically, the EMNIST "Digits" subset contains handwritten digit images, similar to the original MNIST dataset. The number of images in the EMNIST "Digits" subset is as follows:

Training set: The training set of EMNIST "Digits" contains 240,000 images.
Test set: The test set of EMNIST "Digits" contains 40,000 images.
Therefore, the total number of images in the EMNIST "Digits" subset is 280,000 (240,000 training images + 40,000 test images).

The images in EMNIST "Digits" have the same format as the original MNIST dataset, with each image being a grayscale image of size 28x28 pixels, centered and normalized.

The SVHN (Street View House Numbers) dataset is a real-world image dataset for developing machine learning and object recognition algorithms with a focus on recognizing digits and numbers in natural scene images. The dataset was obtained from house numbers in Google Street View images.

The SVHN dataset consists of two formats:

Original Images:
Training set: 73,257 images
Test set: 26,032 images
Extra training set: 531,131 images
The original images are in color and of varying sizes, containing one or more digits in each image.
MNIST-like 32x32 images centered around a single character:
Training set: 73,257 images
Test set: 26,032 images
Extra training set: 531,131 images
These images are preprocessed to be similar to the MNIST dataset format, with each image cropped to 32x32 pixels and centered around a single digit.
In total, the SVHN dataset contains 630,420 images (73,257 + 26,032 + 531,131) in both the original format and the MNIST-like format.

The USPS (United States Postal Service) dataset is a collection of handwritten digit images scanned from envelopes by the U.S. Postal Service. It is commonly used as a benchmark dataset for digit recognition tasks in machine learning and computer vision.

The USPS dataset contains a total of 9,298 images. These images are divided into two subsets:

Training set: The training set consists of 7,291 images.
Test set: The test set contains the remaining 2,007 images.
Each image in the USPS dataset is a grayscale image with a size of 16x16 pixels. The digits in the images are centered and normalized to fit within the 16x16 pixel frame.

Compared to the MNIST dataset, the USPS dataset has a smaller number of images and a lower resolution. However, it provides a different set of handwritten digit samples and can be used as an additional benchmark or for testing the generalization ability of digit recognition models trained on other datasets like MNIST.

The Semeion dataset, also known as the Semeion Handwritten Digit dataset, is a collection of handwritten digit images created by the Semeion Research Center of Sciences of Communication in Rome, Italy. It is used for benchmarking and evaluating handwritten digit recognition models.

The Semeion dataset contains a total of 1,593 handwritten digit images. Each image is a grayscale image with a size of 16x16 pixels, similar to the USPS dataset.

The dataset is structured as follows:

The dataset consists of 1,593 rows, where each row represents a single handwritten digit image.
Each row contains 256 columns representing the pixel values of the 16x16 image, followed by an additional 10 columns representing the one-hot encoded label of the digit (0-9).
Unlike some other well-known datasets like MNIST or USPS, the Semeion dataset does not have a predefined split into training and test sets. Researchers often use their own split ratios or cross-validation techniques when working with this dataset.
"""

import os
import shutil
import torch
import torchvision.transforms as transforms
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import Dataset
from PIL import Image

# Define path
drive_path = "/content/drive/My Drive/"
assert drive_path.endswith("/")

# Initialize variables
train_set, test_set = None, None

# Dataset logic
if dataset_name == 'MNIST':
    data_dir = os.path.join(drive_path, 'data/MNIST/')

    # NO data augmentation for clean model - apply defense at TEST TIME only
    train_transform = ToTensor()

    train_set = datasets.MNIST(data_dir, train=True, download=True, transform=train_transform)
    test_set = datasets.MNIST(data_dir, train=False, download=True, transform=ToTensor())

elif dataset_name == 'EMNIST':
    from torchvision.datasets import EMNIST
    data_dir = os.path.join(drive_path, 'data/EMNIST/raw/')

    # NO data augmentation for clean model - apply defense at TEST TIME only
    train_transform = ToTensor()

    train_set = EMNIST(data_dir, split='digits', train=True, download=True, transform=train_transform)
    test_set = EMNIST(data_dir, split='digits', train=False, download=True, transform=ToTensor())

elif dataset_name == 'SVHN':
    data_dir = os.path.join(drive_path, 'data/SVHN/')
    svhn_transform = transforms.Compose([
        transforms.Resize((28, 28)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor()
    ])
    train_set = datasets.SVHN(data_dir, split='train', download=True, transform=svhn_transform)
    test_set = datasets.SVHN(data_dir, split='test', download=True, transform=svhn_transform)

elif dataset_name == 'USPS':
    data_dir = os.path.join(drive_path, 'data/USPS/')
    usps_transform = transforms.Compose([
        transforms.Resize((28, 28)),
        transforms.ToTensor()
    ])
    train_set = datasets.USPS(data_dir, train=True, download=True, transform=usps_transform)
    test_set = datasets.USPS(data_dir, train=False, download=True, transform=usps_transform)

elif dataset_name == 'TrafficSigns':
    annotation_dir = "/content/drive/My Drive/datasets/StopSigns/annotations"
    image_dir = "/content/drive/My Drive/datasets/StopSigns/images"
    wordlist_path = "/content/drive/My Drive/datasets/StopSigns/wordlist.txt"

    label_map = build_traffic_sign_label_map(annotation_dir)
    class_names = save_traffic_sign_wordlist(label_map, wordlist_path)
    label_to_idx = {label: idx for idx, label in enumerate(class_names)}

    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])
    dataset = TrafficSignsDataset(image_dir, label_map, label_to_idx, transform)
    dataset.label_to_idx = label_to_idx  # attach for downstream compatibility
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size

    # FIX: Use fixed random seed for consistent train/test split across runs
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    train_set, test_set = torch.utils.data.random_split(dataset, [train_size, test_size], generator=generator)
    print(f"TrafficSigns: Using fixed random seed ({RANDOM_SEED}) for consistent 80/20 split")

elif dataset_name == 'ReflectSign':
    reflectsign_df = prepare_reflectsign_dataset()
    from torchvision import transforms
    import torch
    import os
    from PIL import Image

    # === Paths
    base_dir = "/content/drive/MyDrive/datasets/SignRetroreflectivity"
    train_dir = os.path.join(base_dir, "Train Img")
    test_dir = os.path.join(base_dir, "Test Img")
    wordlist_path = os.path.join(base_dir, "wordlist.txt")

    # === Load wordlist.txt → filename_to_label and label_to_idx
    filename_to_label = {}
    with open(wordlist_path, "r") as f:
        for line in f:
            fname, label = line.strip().split(" ", 1)
            filename_to_label[fname] = label

    # Build unique class names and index mapping
    all_labels = sorted(set(filename_to_label.values()))
    label_to_idx = {label: idx for idx, label in enumerate(all_labels)}

    #  Define Dataset Class
    class ReflectSignDataset(torch.utils.data.Dataset):
        def __init__(self, image_dir, filename_to_label, label_to_idx, transform=None):
            self.image_dir = image_dir
            self.transform = transform
            self.filename_to_label = filename_to_label
            self.label_to_idx = label_to_idx

            # Filter only valid files
            self.filenames = [
                f for f in os.listdir(image_dir)
                if f.endswith(".png") and f in filename_to_label and filename_to_label[f] in label_to_idx
            ]
            self.labels = [label_to_idx[filename_to_label[f]] for f in self.filenames]

        def __len__(self):
            return len(self.filenames)

        def __getitem__(self, idx):
            img_path = os.path.join(self.image_dir, self.filenames[idx])
            image = Image.open(img_path).convert("RGB")
            label = self.labels[idx]
            if self.transform:
                image = self.transform(image)
            return image, label

    #  Define Transform
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])

    #  Create train/test datasets
    train_set = ReflectSignDataset(train_dir, filename_to_label, label_to_idx, transform=transform)
    test_set = ReflectSignDataset(test_dir, filename_to_label, label_to_idx, transform=transform)

    #  Add `.targets` for compatibility with PyTorch APIs
    train_set.targets = torch.tensor(train_set.labels)
    test_set.targets = torch.tensor(test_set.labels)


elif dataset_name == 'TinyImageNet':
    zip_url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
    local_zip = "/content/tiny-imagenet-200.zip"
    unzip_dir = "/content/tiny-imagenet-200"
    drive_target_dir = os.path.join(drive_path, "data/tiny-imagenet-200")

    if not os.path.exists(local_zip):
        print("Downloading TinyImageNet...")
        os.system(f"wget {zip_url} -O {local_zip}")

    if not os.path.exists(unzip_dir):
        print("Unzipping TinyImageNet...")
        os.system(f"unzip -q {local_zip} -d /content/")

    if not os.path.exists(drive_target_dir):
        print("Copying TinyImageNet to Google Drive...")
        shutil.copytree(unzip_dir, drive_target_dir)

    val_src = os.path.join(unzip_dir, "val")
    val_dst = os.path.join(drive_target_dir, "val")
    if not os.path.exists(val_dst) and os.path.exists(val_src):
        shutil.copytree(val_src, val_dst)

    words_src = os.path.join(unzip_dir, "words.txt")
    words_dst = os.path.join(drive_target_dir, "words.txt")
    if os.path.exists(words_src):
        shutil.copy(words_src, words_dst)

    val_dir = os.path.join(drive_target_dir, "val")
    img_dir = os.path.join(val_dir, "images")
    ann_file = os.path.join(val_dir, "val_annotations.txt")
    if os.path.exists(ann_file) and os.path.exists(img_dir):
        print("Reorganizing validation images...")
        with open(ann_file, "r") as f:
            for line in f:
                filename, class_name = line.strip().split('\t')[:2]
                class_dir = os.path.join(val_dir, class_name)
                os.makedirs(class_dir, exist_ok=True)
                src = os.path.join(img_dir, filename)
                dst = os.path.join(class_dir, filename)
                if os.path.exists(src):
                    shutil.move(src, dst)
        shutil.rmtree(img_dir)

    # Training transform with AGGRESSIVE augmentation for 90/80 target
    tiny_train_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(15),
        transforms.RandomCrop(64, padding=8),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
        transforms.RandomGrayscale(p=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))  # ImageNet stats
    ])

    # Test transform without augmentation but same normalization
    tiny_test_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))  # ImageNet stats
    ])

    train_dir = os.path.join(drive_target_dir, 'train')
    val_dir = os.path.join(drive_target_dir, 'val')
    train_set = datasets.ImageFolder(train_dir, transform=tiny_train_transform)
    test_set = datasets.ImageFolder(val_dir, transform=tiny_test_transform)

    # CRITICAL FIX: Ensure train and val use SAME class ordering
    # ImageFolder sorts classes alphabetically per directory, but train/val may differ
    # This causes train to use different classes than test even with same numeric labels
    print("🔧 Synchronizing train/val class mappings...")
    train_class_to_idx = train_set.class_to_idx  # Use train as canonical mapping

    # Remap test_set samples to use train_set's class indices
    new_test_samples = []
    skipped_classes = set()
    for path, old_idx in test_set.samples:
        # Get the class name (WNID) from the path
        class_name = os.path.basename(os.path.dirname(path))
        # Look up the correct index from train_set's mapping
        if class_name in train_class_to_idx:
            new_idx = train_class_to_idx[class_name]
            new_test_samples.append((path, new_idx))
        else:
            # Class doesn't exist in train set
            skipped_classes.add(class_name)

    if skipped_classes:
        print(f"⚠️  Skipped {len(skipped_classes)} classes not in train set: {list(skipped_classes)[:5]}")

    # Update test_set to use train_set's class ordering
    test_set.samples = new_test_samples
    test_set.targets = [s[1] for s in new_test_samples]
    test_set.class_to_idx = train_class_to_idx
    test_set.classes = train_set.classes
    print(f"✓ Test set remapped: {len(test_set.samples)} samples using train class ordering")

    label_map = {i: class_name for i, class_name in enumerate(train_set.classes)}
    class_indices = list(label_map.keys())

    class_names = sorted(set(label_map.values()))
    label_to_idx = {label: idx for idx, label in enumerate(class_names)}
    class_map = {orig: new for new, orig in enumerate(class_indices)}

    train_set.samples = [s for s in train_set.samples if s[1] in class_indices]
    test_set.samples = [s for s in test_set.samples if s[1] in class_indices]

    train_set.targets = torch.tensor([class_map[s[1]] for s in train_set.samples])
    test_set.targets = torch.tensor([class_map[s[1]] for s in test_set.samples])

else:
    raise ValueError("Invalid dataset name selected.")

print("Dataset loaded successfully.")

from torch.utils.data import Subset
import torch
import numpy as np

def filtered_dataset(train_set, test_set, filtered=0, dataset_name=None):

    # Wrap in Subset if not already a Subset
    if not isinstance(train_set, Subset):
        train_set = Subset(train_set, list(range(len(train_set))))
    if not isinstance(test_set, Subset):
        test_set = Subset(test_set, list(range(len(test_set))))

    # Extract targets for ALL datasets when they're Subset objects
    if not hasattr(train_set, 'targets'):
        print("Extracting training labels...")
        if hasattr(train_set.dataset, 'samples'):
            # For ImageFolder-style datasets (TinyImageNet, TrafficSigns)
            train_targets = [train_set.dataset.samples[idx][1] for idx in train_set.indices]
        elif hasattr(train_set.dataset, 'targets'):
            # For standard PyTorch datasets (MNIST, CIFAR, etc.)
            base_targets = train_set.dataset.targets
            if isinstance(base_targets, list):
                train_targets = [base_targets[idx] for idx in train_set.indices]
            else:
                train_targets = base_targets[train_set.indices].tolist()
        elif hasattr(train_set.dataset, 'labels'):
            # For datasets using 'labels' instead of 'targets'
            base_labels = train_set.dataset.labels
            if isinstance(base_labels, list):
                train_targets = [base_labels[idx] for idx in train_set.indices]
            else:
                train_targets = base_labels[train_set.indices].tolist()
        else:
            # Last resort: iterate through the dataset
            train_targets = [train_set.dataset[idx][1] for idx in train_set.indices]
        train_set.targets = torch.tensor(train_targets)
        print(f"Training labels extracted: {len(train_targets)} samples")

    if not hasattr(test_set, 'targets'):
        print("Extracting test labels...")
        if hasattr(test_set.dataset, 'samples'):
            test_targets = [test_set.dataset.samples[idx][1] for idx in test_set.indices]
        elif hasattr(test_set.dataset, 'targets'):
            base_targets = test_set.dataset.targets
            if isinstance(base_targets, list):
                test_targets = [base_targets[idx] for idx in test_set.indices]
            else:
                test_targets = base_targets[test_set.indices].tolist()
        elif hasattr(test_set.dataset, 'labels'):
            base_labels = test_set.dataset.labels
            if isinstance(base_labels, list):
                test_targets = [base_labels[idx] for idx in test_set.indices]
            else:
                test_targets = base_labels[test_set.indices].tolist()
        else:
            test_targets = [test_set.dataset[idx][1] for idx in test_set.indices]
        test_set.targets = torch.tensor(test_targets)
        print(f"Test labels extracted: {len(test_targets)} samples")

    # Set train/test labels
    if dataset_name in ['SVHN', 'EMNIST']:
        train_labels = train_set.targets if hasattr(train_set, 'targets') else torch.tensor(train_set.labels)
        test_labels = test_set.targets if hasattr(test_set, 'targets') else torch.tensor(test_set.labels)
    elif dataset_name == 'USPS':
        train_labels = torch.tensor(train_set.targets) if not isinstance(train_set.targets, torch.Tensor) else train_set.targets
        test_labels = torch.tensor(test_set.targets) if not isinstance(test_set.targets, torch.Tensor) else test_set.targets
    else:
        train_labels = train_set.targets if isinstance(train_set.targets, torch.Tensor) else torch.tensor(train_set.targets)
        test_labels = test_set.targets if isinstance(test_set.targets, torch.Tensor) else torch.tensor(test_set.targets)

    if filtered == 0:
        # Binary classification (class 0 and 1)
        print("Sampling for binary classification (classes 0 and 1)...")

        train_indexes_0 = torch.where(train_labels == 0)[0]
        train_indexes_1 = torch.where(train_labels == 1)[0]
        test_indexes_0 = torch.where(test_labels == 0)[0]
        test_indexes_1 = torch.where(test_labels == 1)[0]

        num_to_sample_train = min(len(train_indexes_0), len(train_indexes_1))
        num_to_sample_test = min(len(test_indexes_0), len(test_indexes_1))

        sampled_train_indexes_0 = train_indexes_0[torch.randperm(len(train_indexes_0))[:num_to_sample_train]]
        sampled_train_indexes_1 = train_indexes_1[torch.randperm(len(train_indexes_1))[:num_to_sample_train]]
        sampled_test_indexes_0 = test_indexes_0[torch.randperm(len(test_indexes_0))[:num_to_sample_test]]
        sampled_test_indexes_1 = test_indexes_1[torch.randperm(len(test_indexes_1))[:num_to_sample_test]]

        selected_train_indexes = torch.cat([sampled_train_indexes_0, sampled_train_indexes_1])
        selected_test_indexes = torch.cat([sampled_test_indexes_0, sampled_test_indexes_1])

    elif filtered == 1:
        # Multi-class
        num_classes_total = len(set(train_labels.tolist()))

        # CRITICAL FIX: For TinyImageNet, select 5 DIVERSE classes
        if dataset_name == 'TinyImageNet':
            # Get all unique classes
            all_classes = sorted(set(train_labels.tolist()))

            # TinyImageNet WordNet ordering: early classes are heavily animal-focused
            # With 97 total classes, skip first 50 to definitely avoid animals
            # Select 5 classes from the remaining 47
            n = len(all_classes)
            skip_count = min(50, int(n * 0.5))  # Skip first 50 or 50%
            remaining = all_classes[skip_count:]

            if len(remaining) >= 5:
                step = max(1, len(remaining) // 5)
                selected_classes = [remaining[min(i * step, len(remaining)-1)] for i in range(5)]
            else:
                # Fallback: use last 5 if we skipped too many
                selected_classes = all_classes[-5:]

            # Store globally for label remapping wrapper
            global tinyimagenet_selected_classes
            tinyimagenet_selected_classes = selected_classes

            print(f"TinyImageNet: Skipping first {skip_count} classes to avoid animals")
            print(f"Total classes: {n}, Selecting from last {len(remaining)} classes")
            print(f"Selected class indices: {selected_classes}")
            print(f"DEBUG: First few of all_classes: {all_classes[:10]}")
            print(f"DEBUG: Last few of all_classes: {all_classes[-10:]}")

            # Filter to only selected classes (KEEP ORIGINAL LABELS)
            train_mask = sum(train_labels == idx for idx in selected_classes).bool()
            test_mask = sum(test_labels == idx for idx in selected_classes).bool()

            train_idx_filtered = torch.where(train_mask)[0]
            test_idx_filtered = torch.where(test_mask)[0]

            # Keep original labels (DON'T remap yet - wrapper will do it later)
            train_labels = train_labels[train_mask]
            test_labels = test_labels[test_mask]

            num_classes = 5
        else:
            num_classes = num_classes_total

        print(f"Sampling for multi-class classification ({num_classes} classes)...")

        train_indexes = []
        test_indexes = []

        # For TinyImageNet, search the ORIGINAL dataset for selected classes
        if dataset_name == 'TinyImageNet':
            # Use dataset.targets if available (ImageFolder datasets have this)
            # This is MUCH faster than loading every image
            print("Getting labels from dataset...")
            if hasattr(train_set, 'targets'):
                full_train_labels = torch.tensor(train_set.targets)
                full_test_labels = torch.tensor(test_set.targets)
            elif hasattr(train_set, 'imgs'):
                # ImageFolder stores as list of (path, label) tuples
                full_train_labels = torch.tensor([img[1] for img in train_set.imgs])
                full_test_labels = torch.tensor([img[1] for img in test_set.imgs])
            else:
                # Fallback: extract from samples
                full_train_labels = torch.tensor([s[1] for s in train_set.samples])
                full_test_labels = torch.tensor([s[1] for s in test_set.samples])

            print(f"[OK] Got {len(full_train_labels)} train and {len(full_test_labels)} test labels")

            for i, original_class in enumerate(tinyimagenet_selected_classes):
                print(f"Filtering class {i} (original label {original_class})...")
                # Find ALL samples in original dataset with this class label
                train_idx = torch.where(full_train_labels == original_class)[0]
                test_idx = torch.where(full_test_labels == original_class)[0]
                print(f"  Found {len(train_idx)} train and {len(test_idx)} test samples")
                train_indexes.append(train_idx)
                test_indexes.append(test_idx)
        else:
            for i in range(num_classes):
                print(f"Filtering class {i}...")
                if dataset_name == 'Semeion':
                    train_indices_loop = train_set.indices if hasattr(train_set, "indices") else range(len(train_set))
                    test_indices_loop = test_set.indices if hasattr(test_set, "indices") else range(len(test_set))

                    train_idx = torch.tensor([idx for idx in train_indices_loop if train_set[idx][1][i] == 1])
                    test_idx = torch.tensor([idx for idx in test_indices_loop if test_set[idx][1][i] == 1])
                else:
                    train_idx = torch.where(train_labels == i)[0]
                    test_idx = torch.where(test_labels == i)[0]

                train_indexes.append(train_idx)
                test_indexes.append(test_idx)

        # Sample each class independently for TinyImageNet
        if dataset_name == 'TinyImageNet':
            sampled_train_indexes = []
            sampled_test_indexes = []
            for train_idx, test_idx in zip(train_indexes, test_indexes):
                # Dataset has ~500 train and ~50 test per class available
                # Using ALL 500 train, 50 test per class for 80%+ accuracy target
                # Total: 2500 train, 250 test
                n_train = min(len(train_idx), 500)
                n_test = min(len(test_idx), 50)
                sampled_train_indexes.append(train_idx[torch.randperm(len(train_idx))[:n_train]])
                sampled_test_indexes.append(test_idx[torch.randperm(len(test_idx))[:n_test]])
            print(f"TinyImageNet: {n_train} train and {n_test} test per class")
            print(f"Total: {num_classes * n_train} training, {num_classes * n_test} test images")
            print(f"Using ALL available data for 80%+ accuracy target")
        else:
            min_num_samples_train = min(len(idx) for idx in train_indexes)
            min_num_samples_test = min(len(idx) for idx in test_indexes)

            # Apply AGGRESSIVE limits for MNIST and EMNIST for FAST experiments
            if dataset_name in ['MNIST', 'EMNIST']:
                # Limit to 200 training samples per class (2000 total for 10 classes)
                # Limit to 50 test samples per class (500 total for 10 classes)
                min_num_samples_train = min(min_num_samples_train, 200)
                min_num_samples_test = min(min_num_samples_test, 50)
                print(f"{dataset_name}: Using FAST subset - {min_num_samples_train} train and {min_num_samples_test} test per class")
                print(f"Total: {min_num_samples_train * num_classes} training, {min_num_samples_test * num_classes} test images")

            # CRITICAL FIX: For TrafficSigns/TinyImageNet, don't over-reduce if one class is small
            # Use ALL available samples if min is too low to maintain good accuracy
            if dataset_name in ['TrafficSigns', 'TinyImageNet', 'ReflectSign']:
                if min_num_samples_train < 100:
                    print(f"WARNING: Smallest class has only {min_num_samples_train} training samples")
                    print(f"Using UNBALANCED dataset to preserve training data and accuracy")
                    # Use all available samples for each class (no balancing)
                    sampled_train_indexes = train_indexes
                    sampled_test_indexes = test_indexes
                else:
                    print(f"{dataset_name}: Using balanced subset - {min_num_samples_train} train and {min_num_samples_test} test per class")
                    sampled_train_indexes = [
                        idx[torch.randperm(len(idx))[:min_num_samples_train]] for idx in train_indexes
                    ]
                    sampled_test_indexes = [
                        idx[torch.randperm(len(idx))[:min_num_samples_test]] for idx in test_indexes
                    ]
            else:
                sampled_train_indexes = [
                    idx[torch.randperm(len(idx))[:min_num_samples_train]] for idx in train_indexes
                ]
                sampled_test_indexes = [
                    idx[torch.randperm(len(idx))[:min_num_samples_test]] for idx in test_indexes
                ]

        selected_train_indexes = torch.cat(sampled_train_indexes)
        selected_test_indexes = torch.cat(sampled_test_indexes)

    else:
        raise ValueError("Invalid value for 'filtered'. Choose either 0 or 1.")

    filtered_train_set = Subset(train_set, selected_train_indexes)
    filtered_test_set = Subset(test_set, selected_test_indexes)

    # TinyImageNet-specific: Remap labels to 0-(num_classes-1)
    if dataset_name == 'TinyImageNet' and filtered == 1:
        label_mapping = {old: new for new, old in enumerate(tinyimagenet_selected_classes)}

        class RemapLabels(torch.utils.data.Dataset):
            def __init__(self, dataset, label_map):
                self.dataset = dataset
                self.label_map = label_map

            def __len__(self):
                return len(self.dataset)

            def __getitem__(self, idx):
                img, label = self.dataset[idx]
                return img, self.label_map[label]

        filtered_train_set = RemapLabels(filtered_train_set, label_mapping)
        filtered_test_set = RemapLabels(filtered_test_set, label_mapping)
        print(f"[OK] Labels remapped: {tinyimagenet_selected_classes} → [0, 1, 2, 3, 4]")

    print("Dataset filtering complete.")
    return filtered_train_set, filtered_test_set

from torch.utils.data import DataLoader
import os
import torch

print("Dataset selected:", dataset_name)

# Step 1: Filter dataset by class balance
print("Step 1: Filtering dataset by class balance...")
if dataset_name == 'EMNIST':
    filtered_train_set, filtered_test_set = filtered_dataset_EMNIST(train_set, test_set)

else:
    filtered_train_set, filtered_test_set = filtered_dataset(
        train_set, test_set, filtered=1, dataset_name=dataset_name
)

# Fallback: Check if dataset is empty
if len(filtered_train_set) == 0 or len(filtered_test_set) == 0:
    print("Filtered dataset is empty — using unfiltered datasets instead.")
    filtered_train_set = train_set
    filtered_test_set = test_set
else:
    print("Step 1 complete: Filtered dataset created.")

# Step 2: Build loaders
print("Step 2: Creating DataLoaders...")
batch_size = 256 if dataset_name == 'TinyImageNet' else 64  # Increased for faster training
filtered_train_loader = DataLoader(filtered_train_set, batch_size=batch_size, shuffle=True)
filtered_test_loader = DataLoader(filtered_test_set, batch_size=batch_size)
print("Step 2 complete: DataLoaders ready.")

# Summary
print("Filtered Training Batches:", len(filtered_train_loader))
print("Filtered Test Batches:", len(filtered_test_loader))
print("Filtered Training Images:", len(filtered_train_set))
print("Filtered Test Images:", len(filtered_test_set))

# Step 3: Preview filenames
if dataset_name in ['TinyImageNet', 'TrafficSigns', 'ReflectSign']:
    if hasattr(filtered_train_set, 'indices'):
        print("Step 3: Previewing sample image filenames...")
        sample_indices = filtered_train_set.indices[:5]

        # TinyImageNet case
        if dataset_name == 'TinyImageNet':
            if hasattr(train_set, 'samples'):
                sample_paths = [train_set.samples[i][0] for i in sample_indices]
                print("TinyImageNet sample training image filenames:")
                for path in sample_paths:
                    print("-", os.path.basename(path))
            else:
                print("train_set.dataset has no attribute 'samples'")

        # TrafficSigns case
        elif dataset_name in ['TrafficSigns']:
            if hasattr(train_set.dataset, 'filenames'):
                sample_filenames = [train_set.dataset.filenames[i] for i in sample_indices]
                print("TrafficSigns sample training image filenames:")
                for fname in sample_filenames:
                    print("-", fname)
            else:
                print("train_set.dataset has no attribute 'filenames'")

        elif dataset_name == 'ReflectSign':
            filtered_train_set, filtered_test_set = train_set, test_set

    else:
        print("Warning: filtered_train_set is not a Subset — cannot extract filenames.")

import os
import torch

def build_label_map(dataset_name, dataset, drive_path=None):
    """
    Builds a label map for supported datasets.
    Supports: TinyImageNet, TrafficSigns, default PyTorch datasets.
    """

    if dataset_name.lower() == "tinyimagenet":
        # TinyImageNet: use words.txt for human-readable labels
        words_txt_path = os.path.join(drive_path, "data/tiny-imagenet-200", "words.txt")
        wnids_path = os.path.join(drive_path, "data/tiny-imagenet-200", "wnids.txt")

        synset_to_label = {}
        if os.path.exists(words_txt_path):
            with open(words_txt_path, 'r') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) == 2:
                        synset, label = parts
                        synset_to_label[synset] = label.split(',')[0]
        else:
            print("'words.txt' not found. Using class_to_idx fallback.")
            return {v: k for k, v in dataset.class_to_idx.items()}

        label_map = {}
        for synset, class_idx in dataset.class_to_idx.items():
            readable_label = synset_to_label.get(synset, synset)
            label_map[class_idx] = readable_label

        return label_map

    elif dataset_name.lower() == "trafficsigns":
        # Handle Subset case (e.g., from random_split)
        if isinstance(dataset, torch.utils.data.Subset):
            dataset = dataset.dataset

        # Use label_to_idx if present
        if hasattr(dataset, "label_to_idx"):
            reverse_map = {v: k for k, v in dataset.label_to_idx.items()}
            return {i: reverse_map[i] for i in sorted(reverse_map)}
        else:
            print("Missing label_to_idx in TrafficSigns dataset.")
            return {i: f"Class_{i}" for i in range(10)}

    elif hasattr(dataset, "classes") and isinstance(dataset.classes, list):
        return {i: str(cls) for i, cls in enumerate(dataset.classes)}

    else:
        return {i: str(i) for i in range(10)}

def image_quilting_transform(x):
    return apply_image_quilting(x)

def logit_pairing_transform(model):
    return lambda x: apply_adversarial_logit_pairing(model, x)

def gaussian_noise_transform(x):
    return apply_gaussian_noise_defense(x)

def combined_input_transform(model):
    return lambda x: apply_combined_input_transformation(model, x, labels=None)

def denormalize(tensor, mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)):
    """Reverse normalization for visualization (in-place safe)."""
    if tensor.ndim == 3:
        for t, m, s in zip(tensor, mean, std):
            t.mul_(s).add_(m)
    return tensor

import matplotlib.pyplot as plt

def print_images(imgs, labels, preds, n=3, label_map=None):
    plt.figure(figsize=(12, 4))
    for i in range(n):
        img = imgs[i].clone()
        is_grayscale = img.shape[0] == 1

        # Denormalize based on channels
        if is_grayscale:
            img = denormalize(img, mean=(0.5,), std=(0.5,))
            img = img.squeeze(0)  # (1, H, W) → (H, W)
            cmap = 'gray'
        else:
            img = denormalize(img, mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
            img = img.permute(1, 2, 0)  # (C, H, W) → (H, W, C)
            cmap = None

        img_np = img.numpy()
        img_np = np.clip(img_np, 0, 1)

        # Plot
        plt.subplot(1, n, i + 1)
        plt.imshow(img_np, cmap=cmap)
        label_text = label_map[labels[i].item()] if label_map else labels[i].item()
        pred_text = label_map[preds[i].item()] if label_map else preds[i].item()
        plt.title(f"Orig: {label_text}\nPred: {pred_text}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

def save_clean_model(model, save_dir, filename="clean_model.pth"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    model_path = os.path.join(save_dir, filename)
    torch.save(model.state_dict(), model_path)

import matplotlib.pyplot as plt

def plot_accuracy_loss(train_accuracies, train_losses, val_accuracies=None, val_losses=None):
    epochs = range(1, len(train_accuracies) + 1)

    plt.figure(figsize=(12, 5))  # Slightly more compact

    # Accuracy Plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_accuracies, 'b-o', label='Train Accuracy')
    if val_accuracies is not None and len(val_accuracies) > 0:
        plt.plot(epochs, val_accuracies, 'r-o', label='Val Accuracy')
    plt.title('Model Accuracy over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True)
    plt.legend()

    # Loss Plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_losses, 'b-o', label='Train Loss')
    if val_losses is not None and len(val_losses) > 0:
        plt.plot(epochs, val_losses, 'r-o', label='Val Loss')
    plt.title('Model Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
import torch

def load_tinyimagenet_subset(train_dir, val_dir, selected_classes=None, batch_size=64):
    """
    Loads TinyImageNet dataset, filters to selected class indices, and returns DataLoaders.
    """
    # Transform to match MNIST input: 1x28x28
    # Training transform with augmentation
    tiny_train_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(64, padding=8),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    # Test transform without augmentation
    tiny_test_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    # Load full datasets
    full_train_set = ImageFolder(train_dir, transform=tiny_transform)
    full_test_set = ImageFolder(val_dir, transform=tiny_transform)

    if selected_classes is None:
        selected_classes = list(range(10))  # Default to first 10 classes

    # Build class map
    class_map = {original: new for new, original in enumerate(selected_classes)}

    def filter_and_remap(dataset, class_map):
        indices = [i for i, (_, label) in enumerate(dataset) if label in class_map]
        subset = Subset(dataset, indices)

        class RemappedDataset(torch.utils.data.Dataset):
            def __init__(self, subset, class_map):
                self.subset = subset
                self.class_map = class_map

            def __len__(self):
                return len(self.subset)

            def __getitem__(self, idx):
                img, label = self.subset[idx]
                return img, self.class_map[label]

        return RemappedDataset(subset, class_map)

    # Apply remapping
    filtered_train_set = filter_and_remap(full_train_set, class_map)
    filtered_test_set = filter_and_remap(full_test_set, class_map)

    # DataLoaders
    train_loader = DataLoader(filtered_train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(filtered_test_set, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

from tqdm import tqdm
import os
import time
import torch

def train_model(model, train_loader, loss_fn, optimizer, num_epochs, device,
                batch_size=128, learning_rate=0.001, weight_decay=0.0001,
                label_map=None, dataset_name=None):

    save_dir = '/content/drive/My Drive/clean_models'  # Save to Google Drive
    os.makedirs(save_dir, exist_ok=True)

    train_accuracies = []
    train_losses = []

    model.to(device)
    torch.manual_seed(42)
    print(f"Model running on: {next(model.parameters()).device}")

    # SPEED OPTIMIZATION: Mixed precision training for TinyImageNet (2-3x faster)
    # DISABLED: Causes CUDA errors with small batch sizes
    use_amp = False
    # use_amp = (dataset_name == 'TinyImageNet' and 'cuda' in str(device))
    # if use_amp:
    #     from torch.cuda.amp import autocast, GradScaler
    #     scaler = GradScaler()
    #     print("[OK] Mixed precision training enabled (FP16) - 2-3x speed boost")

    for param_group in optimizer.param_groups:
        param_group['lr'] = learning_rate
        param_group['weight_decay'] = weight_decay

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        #  SWITCH: Only track per-class metrics for ReflectSign
        class_correct = {}
        class_total = {}
        track_per_class = (dataset_name == 'ReflectSign')

        progress_bar = tqdm(train_loader, desc=f"Training Epoch {epoch+1}", unit="batch")

        for i, (imgs, labels) in enumerate(progress_bar):
            batch_start = time.time()

            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            # MIXED PRECISION: Use autocast for forward pass
            if use_amp:
                with autocast():
                    outputs = model(imgs)
                    loss = loss_fn(outputs, labels)

                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(imgs)
                loss = loss_fn(outputs, labels)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            #  SWITCH: Only compute per-class stats for ReflectSign
            if track_per_class and label_map is not None:
                for label_idx, pred_idx in zip(labels.cpu(), predicted.cpu()):
                    label_idx = label_idx.item()
                    pred_idx = pred_idx.item()

                    class_name = label_map.get(label_idx, f"Class_{label_idx}")

                    if class_name not in class_total:
                        class_total[class_name] = 0
                        class_correct[class_name] = 0

                    class_total[class_name] += 1
                    if pred_idx == label_idx:
                        class_correct[class_name] += 1

            if i == 0:
                print("Model on CUDA:", next(model.parameters()).is_cuda)
                print("Image batch device:", imgs.device)
                print_images(imgs.cpu(), labels.cpu(), predicted.cpu(), n=3, label_map=label_map)

        train_loss /= len(train_loader)
        train_acc = train_correct / train_total
        print(f"\nTrain Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}")

        #  SWITCH: Only display per-class accuracy for ReflectSign
        if track_per_class and label_map is not None and len(class_total) > 0:
            print("\n Per-Class Accuracy:")
            for class_name in sorted(class_total.keys()):
                if class_total[class_name] > 0:
                    acc = class_correct[class_name] / class_total[class_name]
                    print(f"  {class_name:40s}: {acc:6.2%} ({class_correct[class_name]:3d}/{class_total[class_name]:3d})")

        train_accuracies.append(train_acc)
        train_losses.append(train_loss)

    plot_accuracy_loss(train_accuracies, train_losses)
    save_clean_model(model, save_dir, filename=f"clean_model_{dataset_name}.pth")
    print(f"[OK] Clean model saved to: {save_dir}/clean_model_{dataset_name}.pth\n")

import os
import time
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

if dataset_name in ['ReflectSign', 'TrafficSigns']:
    if dataset_name == 'ReflectSign':
        # === Setup paths ===
        base_dir = "/content/drive/MyDrive/datasets/SignRetroreflectivity"
        train_img_dir = os.path.join(base_dir, "Train Img")
        wordlist_path = os.path.join(base_dir, "wordlist.txt")

        # === Load filename-to-label mapping ===
        filename_to_label = {}
        with open(wordlist_path, "r") as f:
            for line in f:
                filename, label = line.strip().split(" ", 1)
                filename_to_label[filename] = label
        print(" Loaded wordlist.txt.")

        # === Display first 10 labeled images from Train Img ===
        print("\n Previewing first 10 labeled training images:")
        preview_count = 10
        shown = 0
        for filename, label in filename_to_label.items():
            img_path = os.path.join(train_img_dir, filename)
            if os.path.exists(img_path):
                img = Image.open(img_path).convert("RGB")
                plt.imshow(img)
                plt.title(f"{filename} → {label}")
                plt.axis("off")
                plt.show()
                shown += 1
                if shown >= preview_count:
                    break
            else:
                print(f"[MISSING] Missing image: {filename}")

        #  Build class name → index map
        all_labels = sorted(set(filename_to_label.values()))
        class_to_idx = {label: idx for idx, label in enumerate(all_labels)}
        idx_to_class = {idx: label for label, idx in class_to_idx.items()}
        print(" Class-to-index mapping:", class_to_idx)

        #  Convert filename → label to filename → class index
        filename_to_index = {
            fname: class_to_idx[label]
            for fname, label in filename_to_label.items()
        }

        print("Class-to-index mapping:", class_to_idx)
        print("Number of classes:", len(class_to_idx))

        #  Set label_map for ReflectSign
        label_map = idx_to_class

    elif dataset_name == 'TrafficSigns':
        # Extract label mapping from existing TrafficSigns dataset
        base_dataset = train_set.dataset
        label_to_idx = base_dataset.label_to_idx
        idx_to_class = {idx: label for label, idx in label_to_idx.items()}
        label_map = idx_to_class
        print(f" Extracted {len(label_map)} traffic sign labels")

elif dataset_name == 'TinyImageNet':
    #  TinyImageNet-specific label mapping
    # Get WordNet IDs from dataset
    if hasattr(train_set, 'dataset') and hasattr(train_set.dataset, 'classes'):
        base_classes = train_set.dataset.classes
    elif hasattr(train_set, 'classes'):
        base_classes = train_set.classes
    else:
        base_classes = None

    if base_classes:
        # Load words.txt to get human-readable names
        words_txt_path = '/content/tiny-imagenet-200/words.txt'
        if os.path.exists(words_txt_path):
            wnid_to_words = {}
            with open(words_txt_path, 'r') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) >= 2:
                        wnid_to_words[parts[0]] = parts[1].split(',')[0].strip()

            # Map indices to human-readable names
            idx_to_class = {i: wnid_to_words.get(wnid, wnid) for i, wnid in enumerate(base_classes)}

            # DEBUG: Check if we have selected classes
            print(f"DEBUG: tinyimagenet_selected_classes = {tinyimagenet_selected_classes}")
            print(f"DEBUG: Type = {type(tinyimagenet_selected_classes)}")

            # CRITICAL: If we filtered/remapped classes, update label_map to match
            if tinyimagenet_selected_classes is not None and len(tinyimagenet_selected_classes) > 0:
                print(f"DEBUG: Updating label_map for filtered classes")
                # We remapped classes [48, 57, 66, 75, 84] → [0, 1, 2, 3, 4]
                # Create new label_map with remapped indices
                label_map = {}
                for new_idx, orig_idx in enumerate(tinyimagenet_selected_classes):
                    class_name = idx_to_class.get(orig_idx, f"Class_{orig_idx}")
                    label_map[new_idx] = class_name
                    print(f"DEBUG: Mapping {new_idx} -> {class_name} (orig class {orig_idx})")

                # CRITICAL FIX: Update idx_to_class to match the filtered/remapped label_map
                idx_to_class = label_map

                print(f"[OK] Updated label_map for filtered classes:")
                for new_idx, name in label_map.items():
                    orig_idx = tinyimagenet_selected_classes[new_idx]
                    print(f"  {new_idx}: {name} (original class {orig_idx})")
            else:
                print(f"DEBUG: NOT updating label_map - using default idx_to_class")
                label_map = idx_to_class
                print(f" Loaded {len(wnid_to_words)} human-readable labels from words.txt")
        else:
            # Fallback to WordNet IDs
            idx_to_class = {i: wnid for i, wnid in enumerate(base_classes)}
            label_map = idx_to_class
            print(f"⚠️ words.txt not found, using WordNet IDs")
    else:
        label_map = None
        idx_to_class = None
        print("⚠️ Could not extract classes from TinyImageNet dataset")

else:
    #  Set label_map for other datasets (MNIST, EMNIST, etc.)
    if hasattr(train_set, 'classes'):
        label_map = {i: class_name for i, class_name in enumerate(train_set.classes)}
        idx_to_class = label_map
    elif hasattr(train_set, 'dataset') and hasattr(train_set.dataset, 'classes'):
        label_map = {i: class_name for i, class_name in enumerate(train_set.dataset.classes)}
        idx_to_class = label_map
    else:
        label_map = None
        idx_to_class = None

# === Train the model ===
save_dir = '/content/drive/My Drive/clean_models'  # Save to Google Drive for persistence

# VERIFY GOOGLE DRIVE IS MOUNTED
if not os.path.exists('/content/drive/My Drive'):
    print(f"\n{'='*80}")
    print(f"[ERROR] ERROR: GOOGLE DRIVE NOT MOUNTED")
    print(f"  Please mount Google Drive before running this code:")
    print(f"  from google.colab import drive")
    print(f"  drive.mount('/content/drive')")
    print(f"{'='*80}\n")
    raise RuntimeError("Google Drive not mounted")
else:
    print(f"[OK] Google Drive is mounted")

clean_model_path = f"{save_dir}/clean_model_{dataset_name}.pth"

# CHECK IF CLEAN MODEL ALREADY EXISTS
if os.path.exists(clean_model_path):
    print(f"[OK] Clean model file exists: {clean_model_path}")
    print(f"  Loading existing clean model...\n")

    # Load existing clean model
    if dataset_name in ['ReflectSign', 'TrafficSigns', 'TinyImageNet']:
        class HybridModel(nn.Module):
            def __init__(self, classical_model, theta, phi, device, output_dim):
                super().__init__()
                self.classical_model = classical_model
                self.theta = theta
                self.phi = phi
                self.device = device
                self.output_dim = output_dim

            def forward(self, x):
                return hybrid_forward(x, self.classical_model, self.theta, self.phi, self.device, self.output_dim)

        if dataset_name == "TinyImageNet":
            output_dim = 5  # Changed from 10 to match 5 classes
        elif dataset_name == "TrafficSigns":
            output_dim = 4  # TrafficSigns has 4 classes (crosswalk, speedlimit, stop, trafficlight)
        elif dataset_name == "ReflectSign":
            output_dim = 55
        elif dataset_name in ["MNIST", "EMNIST"]:
            output_dim = 10
        else:
            output_dim = 10
        hybrid_model = HybridModel(classical_model, theta, phi, device, output_dim)

    hybrid_model.load_state_dict(torch.load(clean_model_path))
    hybrid_model.to(device)
    hybrid_model.eval()

    print(f"[OK] Clean model loaded successfully from: {clean_model_path}\n")

else:
    # TRAIN NEW CLEAN MODEL
    print(f"⚠️  Clean model not found. Training new clean model...\n")

    start_time = time.time()

    if dataset_name in ['ReflectSign', 'TrafficSigns', 'TinyImageNet']:
        # Define proper wrapper for hybrid_forward
        class HybridModel(nn.Module):
            def __init__(self, classical_model, theta, phi, device, output_dim):
                super().__init__()
                self.classical_model = classical_model
                self.theta = theta
                self.phi = phi
                self.device = device
                self.output_dim = output_dim

            def forward(self, x):
                return hybrid_forward(x, self.classical_model, self.theta, self.phi, self.device, self.output_dim)

        if dataset_name == "TinyImageNet":
          output_dim = 5  # Changed from 10 to match 5 classes
        elif dataset_name == "TrafficSigns":
          output_dim = 4  # TrafficSigns has 4 classes (crosswalk, speedlimit, stop, trafficlight)
        elif dataset_name == "ReflectSign":
          output_dim = 55
        elif dataset_name in ["MNIST", "EMNIST"]:
          output_dim = 10
        else:
          output_dim = 10
        hybrid_model = HybridModel(classical_model, theta, phi, device, output_dim)

    # TinyImageNet uses global num_epochs setting (120 for increased training data)
    if dataset_name == 'TinyImageNet':
        print(f"TinyImageNet: Using {num_epochs} epochs (with 2000 training images)")

    train_model(
        hybrid_model,
        filtered_train_loader,
        loss_fn,
        optimizer,
        num_epochs,
        device,
        label_map=label_map,
        dataset_name=dataset_name
    )

    end_time = time.time()
    print(f"Training completed in {end_time - start_time:.2f} seconds.")

    # Calculate the training duration in seconds
    training_duration_seconds = abs(end_time - start_time)

    # Calculate the training duration in minutes
    training_duration_minutes = training_duration_seconds / 60

    # Calculate the training duration in hours
    training_duration_hours = training_duration_minutes / 60

    # Print the training duration in different units
    print(f"Training duration: {training_duration_seconds:.2f} seconds")
    print(f"Training duration: {training_duration_minutes:.2f} minutes")
    print(f"Training duration: {training_duration_hours:.2f} hours")

def load_clean_model(model, save_dir, dataset_name):
    # Model folder - include dataset name
    model_path = f"{save_dir}/clean_model_{dataset_name}.pth"

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found in {model_path}")

    model.load_state_dict(torch.load(model_path))
    model.eval()

    print(f"Loaded clean_model state dict from {model_path}")
    return model

num_clean_elements = len(filtered_train_set)
print(f"Number of clean images: {num_clean_elements}")

import matplotlib.pyplot as plt

def plot_test_accuracy_loss(test_accuracy, test_loss):
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.bar(1, test_accuracy, color='r', label='Testing Accuracy')
    plt.title('Testing Accuracy')
    plt.xlabel('Evaluation')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.bar(1, test_loss, color='r', label='Testing Loss')
    plt.title('Testing Loss')
    plt.xlabel('Evaluation')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

from tqdm import tqdm
import torch
from torch.utils.data import TensorDataset, DataLoader

def evaluate(model, data_loader, loss_fn, device):
    test_loss = 0.0
    correct = 0
    total = 0
    model.eval()

    misclassified_images = []
    misclassified_labels = []
    misclassified_preds = []

    with torch.no_grad():
        for x, y in tqdm(data_loader, desc="Evaluating", unit="batch"):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            batch_loss = loss_fn(logits, y)

            test_loss += batch_loss.item() * x.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            incorrect_inds = torch.where(preds != y)[0]
            if len(incorrect_inds) > 0:
                misclassified_images.append(x[incorrect_inds].cpu())
                misclassified_labels.append(y[incorrect_inds].cpu())
                misclassified_preds.append(preds[incorrect_inds].cpu())

    test_loss /= total
    test_accuracy = correct / total
    num_misclassified = total - correct

    print(f"\nNumber of test images: {total}")
    print(f"Misclassified: {num_misclassified}/{total}")
    print(f"Test Loss: {test_loss:.4f} | Accuracy: {test_accuracy:.2%}")

    # Ensure return types are always lists with tensors
    if len(misclassified_images) == 0:
        misclassified_images.append(torch.tensor([]))
        misclassified_labels.append(torch.tensor([]))
        misclassified_preds.append(torch.tensor([]))

    return test_loss, test_accuracy, [torch.cat(misclassified_images)], [torch.cat(misclassified_labels)], [torch.cat(misclassified_preds)]

# Use the hybrid_model as clean_model (already loaded or trained above)
clean_model = hybrid_model
clean_model.eval()  # Put in eval mode

# IMMEDIATE CHECK - Evaluate clean model
clean_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for imgs, labels in filtered_test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = clean_model(imgs)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
print(f" IMMEDIATE EVALUATION: Clean model accuracy on test data: {correct/total:.2%}")

# Then continue with your existing evaluation code
test_loss, test_acc, misclassified_images, misclassified_labels, misclassified_preds = evaluate(clean_model, filtered_test_loader, loss_fn, device)

from torch.utils.data import TensorDataset
# Evaluation
test_loss, test_acc, misclassified_images, misclassified_labels, misclassified_preds = evaluate(clean_model, filtered_test_loader, loss_fn, device)

import math
import torch
import matplotlib.pyplot as plt

def analyze_for_misclassifications(misclassified_images, misclassified_labels, misclassified_preds, label_map=None):
    MAX_IMAGES = 10

    # Convert label_map keys to int if needed
    if label_map and isinstance(next(iter(label_map.keys())), str):
        try:
            label_map = {int(k): v for k, v in label_map.items()}
        except Exception as e:
            print(f"Failed to cast label_map keys to int: {e}")

    # Flatten if stored as lists
    if isinstance(misclassified_images, list):
        misclassified_images = torch.cat(misclassified_images) if misclassified_images else torch.empty(0)
    if isinstance(misclassified_labels, list):
        misclassified_labels = torch.cat(misclassified_labels) if misclassified_labels else torch.empty(0)
    if isinstance(misclassified_preds, list):
        misclassified_preds = torch.cat(misclassified_preds) if misclassified_preds else torch.empty(0)

    if misclassified_images.numel() == 0:
        print("No misclassified examples to display.")
        return misclassified_images, misclassified_labels, misclassified_preds

    print(f"Number misclassified: {len(misclassified_images)}")

    images = misclassified_images[:MAX_IMAGES]
    labels = misclassified_labels[:MAX_IMAGES]
    preds = misclassified_preds[:MAX_IMAGES]
    num_images = len(images)

    rows = math.ceil(num_images / 4)
    cols = min(4, num_images)
    fig = plt.figure(figsize=(3.5 * cols, 3.5 * rows))

    def truncate(text, max_len=22):
        return text[:max_len] + "..." if len(text) > max_len else text

    for i, (img, label, pred) in enumerate(zip(images, labels, preds)):
        ax = fig.add_subplot(rows, cols, i + 1)
        if img.ndim == 2:
            img = img.unsqueeze(0)
        img = img.permute(1, 2, 0)

        # De-normalize assuming Normalize(mean=0.5, std=0.5)
        img = img * 0.5 + 0.5
        img = img.clamp(0, 1)

        label_idx = int(label.item())
        pred_idx = int(pred.item())

        # Get readable class names
        label_name = truncate(label_map[label_idx]) if label_map and label_idx in label_map else str(label_idx)
        pred_name = truncate(label_map[pred_idx]) if label_map and pred_idx in label_map else str(pred_idx)

        ax.imshow(img.squeeze(), cmap='gray' if img.shape[-1] == 1 else None)
        ax.set_title(f"{label_name} → {pred_name}", fontsize=9)
        ax.axis("off")

    plt.subplots_adjust(hspace=0.6)
    plt.tight_layout()
    plt.show()

    return misclassified_images, misclassified_labels, misclassified_preds

if dataset_name == 'ReflectSign':
  # === Display first 10 labeled images from Train Img ===
  print("\n Previewing first 10 labeled training images:")
  preview_count = 10
  shown = 0
  for filename, label in filename_to_label.items():
      img_path = os.path.join(train_img_dir, filename)
      if os.path.exists(img_path):
          img = Image.open(img_path).convert("RGB")
          plt.imshow(img)
          plt.title(f"{filename} → {label}")
          plt.axis("off")
          plt.show()
          shown += 1
          if shown >= preview_count:
              break
      else:
          print(f"[MISSING] Missing image: {filename}")

  #  Build class name → index map
  all_labels = sorted(set(filename_to_label.values()))
  class_to_idx = {label: idx for idx, label in enumerate(all_labels)}
  print(" Class-to-index mapping:", class_to_idx)

  #  Convert filename → label to filename → class index
  filename_to_index = {
      fname: class_to_idx[label]
      for fname, label in filename_to_label.items()
  }

  print("Class-to-index mapping:", class_to_idx)
  print("Number of classes:", len(class_to_idx))

  clean_misclassified_images, clean_misclassified_labels, clean_misclassified_preds = analyze_for_misclassifications(
      misclassified_images,
      misclassified_labels,
      misclassified_preds,
      label_map=idx_to_class
  )

else:

  clean_misclassified_images, clean_misclassified_labels, clean_misclassified_preds = analyze_for_misclassifications(
     misclassified_images,
      misclassified_labels,
      misclassified_preds,
      label_map=label_map
  )

model = clean_model
# Define compounded attack
def compounded_attack(choose_attack_option):
    if choose_attack_option == "fgsm_cw_attack":
        attack1 = torchattacks.FGSM(model, eps=0.3)
        attack2 = torchattacks.CW(model, c=0.1, kappa=0.0, steps=1000)
        attack = torchattacks.MultiAttack([attack1, attack2])
    elif choose_attack_option == "fgsm_pgd_attack":
        attack1 = torchattacks.FGSM(model, eps=0.3)
        attack2 = torchattacks.PGD(model, eps=0.3, alpha=0.01, steps=5)
        attack = torchattacks.MultiAttack([attack1, attack2])
    elif choose_attack_option == "cw_pgd_attack":
        attack1 = torchattacks.CW(model, c=0.1, kappa=0.0, steps=1000)
        attack2 = torchattacks.PGD(model, eps=0.3, alpha=0.01, steps=5)
        attack = torchattacks.MultiAttack([attack1, attack2])
    elif choose_attack_option == "pgd_bim_attack":
        attack1 = torchattacks.PGD(model, eps=0.3, alpha=0.01, steps=5)
        attack2 = torchattacks.BIM(model, eps=8/255, alpha=2/255, steps=10)
        attack = torchattacks.MultiAttack([attack1, attack2])
    elif choose_attack_option == "fgsm_bim_attack":
        attack1 = torchattacks.FGSM(model, eps=0.3)
        attack2 = torchattacks.BIM(model, eps=8/255, alpha=2/255, steps=10)
        attack = torchattacks.MultiAttack([attack1, attack2])
    elif choose_attack_option == "cw_bim_attack":
        attack1 = torchattacks.CW(model, c=0.1, kappa=0.0, steps=1000)
        attack2 = torchattacks.BIM(model, eps=8/255, alpha=2/255, steps=10)
        attack = torchattacks.MultiAttack([attack1, attack2])
    elif choose_attack_option == "pgd_deepfool_attack":
        attack1 = torchattacks.PGD(model, eps=0.3, alpha=0.01, steps=5)
        attack2 = torchattacks.DeepFool(model, steps=50, overshoot=0.02)
        attack = torchattacks.MultiAttack([attack1, attack2])
    elif choose_attack_option == "fgsm_deepfool_attack":
        attack1 = torchattacks.FGSM(model, eps=0.3)
        attack2 = torchattacks.DeepFool(model, steps=50, overshoot=0.02)
        attack = torchattacks.MultiAttack([attack1, attack2])
    elif choose_attack_option == "cw_deepfool_attack":
        attack1 = torchattacks.CW(model, c=0.1, kappa=0.0, steps=1000)
        attack2 = torchattacks.DeepFool(model, steps=50, overshoot=0.02)
        attack = torchattacks.MultiAttack([attack1, attack2])
    elif choose_attack_option == "bim_deepfool_attack":
        attack1 = torchattacks.BIM(model, eps=8/255, alpha=2/255, steps=10)
        attack2 = torchattacks.DeepFool(model, steps=50, overshoot=0.02)
        attack = torchattacks.MultiAttack([attack1, attack2])
    else:
        print("You did not chose any of the possible options or an you made an invalid option")
    return attack

attack = compounded_attack(compounded_attack_name)

import tqdm
import torch
from torch.utils.data import TensorDataset, DataLoader

def perform_adv_attack(model, attack, data_loader, label_map=None, dataset_name=None):
    adv_data = []
    adv_labels = []
    device = next(model.parameters()).device

    total_batches = len(data_loader)
    progress_bar = tqdm.tqdm(total=total_batches, unit='batch', desc=f'⚔️ Adversarial Attack ({dataset_name})' if dataset_name else '⚔️ Adversarial Attack')

    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        y = y.long()  # Ensure labels are of type LongTensor

        try:
            x_adv = attack(x, y)
        except Exception as e:
            print(f"Attack failed on batch: {e}")
            continue

        adv_data.append(x_adv.detach().cpu())
        adv_labels.append(y.detach().cpu())

        progress_bar.update(1)

    progress_bar.close()

    #  Combine all batches into single tensors
    if len(adv_data) == 0:
        print("No adversarial data generated.")
        return None

    adv_data = torch.cat(adv_data)
    adv_labels = torch.cat(adv_labels)

    #  Create new DataLoader for adversarial examples
    adv_dataset = TensorDataset(adv_data, adv_labels)
    adv_loader = DataLoader(adv_dataset, batch_size=64, shuffle=False)

    print(f"Adversarial dataset created with {len(adv_dataset)} samples.")
    return adv_loader

from torch.utils.data import TensorDataset

if dataset_name == 'ReflectSign':

  # Adversarial attack
  adv_attack_loader = perform_adv_attack(
      clean_model,
      attack,
      filtered_test_loader,
      label_map=idx_to_class,              # Optional, used for later display
      dataset_name=dataset_name         # Optional, used for tqdm display
  )

else:
  adv_attack_loader = perform_adv_attack(
      clean_model,
      attack,
      filtered_test_loader,
      label_map=label_map,              # Optional, used for later display
      dataset_name=dataset_name         # Optional, used for tqdm display
  )

import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

def evaluate_adv_attack(model, data_loader, loss_fn, device):
    attack_test_loss = 0
    correct_predictions = 0
    total_samples = 0
    misclassified_images = []
    misclassified_labels = []
    misclassified_preds = []

    model.eval()
    progress_bar = tqdm(data_loader, desc="Evaluating Adversarial Samples", unit="batch")

    with torch.no_grad():
        for x, y in progress_bar:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            attack_test_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct_predictions += (preds == y).sum().item()
            total_samples += y.size(0)

            incorrect_inds = (preds != y).nonzero(as_tuple=True)[0]
            if incorrect_inds.numel() > 0:
                misclassified_images.append(x[incorrect_inds].cpu())
                misclassified_labels.append(y[incorrect_inds].cpu())
                misclassified_preds.append(preds[incorrect_inds].cpu())

    progress_bar.close()

    if total_samples == 0:
        print("Warning: No samples in evaluation.")
        return 0.0, 0.0, torch.empty(0, 3, 28, 28), torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)

    avg_loss = attack_test_loss / len(data_loader)
    accuracy = correct_predictions / total_samples
    print(f"Attack Test Loss: {avg_loss:.4f}")
    print(f"Accuracy on Adversarial Examples: {accuracy:.2%}")
    print(f"Misclassified: {total_samples - correct_predictions} / {total_samples}")

    # Make sure outputs have expected shape even when empty
    example_shape = next(iter(data_loader))[0].shape[1:]  # Get shape from first batch (C, H, W)
    misclassified_images = (
        torch.cat(misclassified_images, dim=0)
        if misclassified_images else torch.empty(0, *example_shape)
    )
    misclassified_labels = (
        torch.cat(misclassified_labels, dim=0)
        if misclassified_labels else torch.empty(0, dtype=torch.long)
    )
    misclassified_preds = (
        torch.cat(misclassified_preds, dim=0)
        if misclassified_preds else torch.empty(0, dtype=torch.long)
    )

    return avg_loss, accuracy, misclassified_images, misclassified_labels, misclassified_preds

# Evaluate adversarial attack
clean_attack_test_loss, clean_attack_adv_test_accuracy, misclassified_images, misclassified_labels, misclassified_preds = evaluate_adv_attack(
    clean_model, adv_attack_loader, loss_fn, device
)

# Summary and heuristic evaluation
print("\nEvaluation Summary of Adversarial Attack:\n")
print(f" Attack Loss: {clean_attack_test_loss:.4f}")
print(f" Attack Accuracy: {clean_attack_adv_test_accuracy:.2%}")
print(f" Misclassified Samples: {len(misclassified_images)}")

print("\nKey Metrics to Evaluate Adversarial Attack Success:")
print("• Significant Drop in Accuracy: Typically a drop of 20–40% or more suggests a strong attack.")
print("• High Loss Value: A higher loss compared to clean evaluation may signal vulnerability.")
print("• Misclassified Examples: Wrongly labeled outputs confirm targeted attack effectiveness.\n")

# Optional: Heuristic indicator of success
if clean_attack_adv_test_accuracy < 0.7:
    print("Attack likely successful (accuracy < 70%).")
elif clean_attack_adv_test_accuracy < 0.9:
    print("Partial success (accuracy dropped, but model still resilient).")
else:
    print("Attack likely weak or model highly robust.")

# Ensure all inputs are tensors before passing to the function
if isinstance(misclassified_images, list):
    misclassified_images = torch.cat(misclassified_images) if misclassified_images else torch.empty(0, 1, 28, 28)
if isinstance(misclassified_labels, list):
    misclassified_labels = torch.cat(misclassified_labels) if misclassified_labels else torch.empty(0, dtype=torch.long)
if isinstance(misclassified_preds, list):
    misclassified_preds = torch.cat(misclassified_preds) if misclassified_preds else torch.empty(0, dtype=torch.long)

if dataset_name == 'ReflectSign':
  # Analyze
  adv_misclassified_images, adv_misclassified_labels, adv_misclassified_preds = analyze_for_misclassifications(
      misclassified_images, misclassified_labels, misclassified_preds, label_map=idx_to_class
  )
else:
  adv_misclassified_images, adv_misclassified_labels, adv_misclassified_preds = analyze_for_misclassifications(
      misclassified_images, misclassified_labels, misclassified_preds, label_map=label_map
  )

import torch
import tqdm
from torch.utils.data import DataLoader, TensorDataset

def generate_adversarial_dataset(model, attack, data_loader, device, num_samples, dataset_name=None, label_map=None):
    adv_data = []
    adv_labels = []
    num_generated = 0
    total_attempts = 0

    device = next(model.parameters()).device
    progress_bar = tqdm.tqdm(total=num_samples, unit='sample', desc='Adversarial Attack')

    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        y = y.long()

        # Enable gradient tracking for adversarial attack
        x.requires_grad_(True)

        total_attempts += y.size(0)

        x_adv = attack(x, y)

        with torch.no_grad():
            orig_preds = model(x).argmax(1)
            adv_preds = model(x_adv).argmax(1)

        mask = orig_preds != adv_preds
        x_adv_filtered = x_adv[mask]
        y_filtered = y[mask]

        if label_map and num_generated == 0 and x_adv_filtered.size(0) > 0:
            for i in range(min(3, len(x_adv_filtered))):
                orig = orig_preds[mask][i].item()
                adv = adv_preds[mask][i].item()
                label = y_filtered[i].item()
                orig_name = label_map.get(orig, str(orig)) if label_map else str(orig)
                adv_name = label_map.get(adv, str(adv)) if label_map else str(adv)
                print(f"[OK] Adversarial success: original '{orig_name}' → adversarial '{adv_name}' (true: '{label_map.get(label, label)}')")

        num_remaining = num_samples - num_generated
        if num_remaining > 0:
            if num_remaining >= len(x_adv_filtered):
                adv_data.append(x_adv_filtered.detach().cpu())
                adv_labels.append(y_filtered.detach().cpu())
                num_generated += len(x_adv_filtered)
                progress_bar.update(len(x_adv_filtered))
            else:
                adv_data.append(x_adv_filtered[:num_remaining].detach().cpu())
                adv_labels.append(y_filtered[:num_remaining].detach().cpu())
                num_generated = num_samples
                progress_bar.update(num_remaining)
                break

        if num_generated >= num_samples:
            break

    progress_bar.close()

    if num_generated == 0:
        print("Warning: No successful adversarial examples were generated.")
        return None

    adv_data = torch.cat(adv_data)
    adv_labels = torch.cat(adv_labels)
    adv_dataset = TensorDataset(adv_data, adv_labels)

    if num_generated < num_samples:
        print(f"Only {num_generated} out of {num_samples} requested adversarial examples generated.")

    success_rate = (num_generated / total_attempts) * 100 if total_attempts else 0
    print(f"Adversarial success rate: {success_rate:.2f}% ({num_generated}/{total_attempts})")

    return adv_dataset

print("Dataset:", dataset_name)

filtered_test_loader = DataLoader(filtered_test_set, batch_size=64)

# Number of batches in the loader
print("Number of batches in the filtered test set:", len(filtered_test_loader))

# Accurate total number of test samples
test_set_size = len(filtered_test_set)
print("Number of images in the filtered test set:", test_set_size)

"""The number of adversarial images used for training as a defense against adversarial attacks on a model follow general guidelines and considerations:

Balanced dataset: It is generally recommended to have a balanced dataset, where the number of adversarial examples is similar to the number of clean examples. This helps the model learn to distinguish between clean and adversarial examples effectively.

Percentage of adversarial examples: A common practice is to include a certain percentage of adversarial examples in the training set. The percentage can range from 10% to 50% of the total training examples, depending on the specific requirements and the sensitivity of the application.

Iterative training: Instead of training with a fixed number of adversarial examples, an iterative approach can be employed. Start with a smaller percentage of adversarial examples and gradually increase the proportion over multiple training iterations. This allows the model to adapt and improve its robustness incrementally.

Diversity of adversarial examples: It is important to generate adversarial examples using different attack methods and parameters to expose the model to a wide range of adversarial perturbations. This helps the model learn to defend against various types of attacks.

Validation and testing: It is crucial to evaluate the model's performance on a separate validation set containing both clean and adversarial examples. This helps assess the model's robustness and generalization ability. The test set should also include a mix of clean and adversarial examples to measure the model's performance in real-world scenarios.

Computational resources: Generating and training with a large number of adversarial examples can be computationally expensive. Consider the available computational resources and time constraints when determining the number of adversarial examples to include in the training process.
"""

# Dynamically adjust adversarial example count by dataset
if dataset_name == 'MNIST' or dataset_name == 'USPS':
    ratio = 0.5
elif dataset_name == 'EMNIST':
    ratio = 0.005
elif dataset_name == 'SVHN':
    ratio = 0.3
elif dataset_name == 'TinyImageNet':
    ratio = 0.1
elif dataset_name == 'TrafficSigns':
    ratio = 0.5
elif dataset_name == 'ReflectSign':
    ratio = 1
else:
    ratio = 0.2  # default fallback for unknown datasets

num_of_adv_examples = int(ratio * len(filtered_train_set))
print(f"Number of adversarial examples to be created: {num_of_adv_examples}")

# Ensure model is on the correct device and in eval mode
model.to(device)
model.eval()

# Confirm dataset being used
print(f"Dataset in use: {dataset_name}")
print(f"Filtered training samples: {len(filtered_train_set)}")

# Skip adversarial generation for randomization and input_transformation
if defense_type in ["randomization", "input_transformation"]:
    print(f"\n{'='*80}")
    print(f"{defense_type.upper()} DEFENSE: Skipping adversarial dataset generation")
    if defense_type == "randomization":
        print("Randomization applies transforms at TEST time only - no training needed")
    else:
        print("Input transformation trains on CLEAN transformed data - no adversarial needed")
    print(f"{'='*80}\n")

    # Create dummy variables for compatibility
    adv_dataset = TensorDataset(torch.empty(0, 1, 28, 28), torch.empty(0, dtype=torch.long))
    adv_loader = DataLoader(adv_dataset, batch_size=1)
    num_adv_images = 0
    num_adv_batches = 0

else:
    # For adversarial_training ONLY: generate adversarial images
    print(f"\nGenerating adversarial dataset for {defense_type} defense...")

    # Number of adversarial examples to generate
    num_of_adv_examples = int(ratio * len(filtered_train_set))
    print(f"Number of adversarial examples to be created: {num_of_adv_examples}")

    # Generate adversarial dataset
    adv_dataset = generate_adversarial_dataset(
        model, attack, filtered_train_loader, device, num_of_adv_examples
    )

    # Ensure dataset generation succeeded
    assert adv_dataset is not None and len(adv_dataset) > 0, "No adversarial examples were generated."

    # Get proper device object if not already
    device_obj = torch.device(device) if isinstance(device, str) else device

    # Create DataLoader with CUDA optimization options
    adv_loader = DataLoader(
        adv_dataset,
        batch_size=len(adv_dataset),
        shuffle=False,
        num_workers=1,
        pin_memory=True if device_obj.type == 'cuda' else False
    )

    # Extract all tensors
    adv_images, adv_labels = next(iter(adv_loader))

    # Rewrap as TensorDataset
    adv_ds = TensorDataset(adv_images, adv_labels)

    # Batch size and batch count
    adv_batch_size = 128
    num_adv_images = len(adv_ds)
    num_adv_batches = (num_adv_images + adv_batch_size - 1) // adv_batch_size

    print(f"Number of adversarial images: {num_adv_images}")
    print(f"Number of batches (with batch size {adv_batch_size}): {num_adv_batches}")

import math
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

def plot_adversarial_images(adv_dataset, model, dataset_name=None, label_map=None, max_images=10):
    # Handle label map conditionally
    if dataset_name in ["MNIST", "EMNIST", "SVHN", "USPS"]:
        if label_map is None:
            label_map = {}  # Use numeric labels
    elif dataset_name in ["TinyImageNet", "TrafficSigns"]:
        if label_map is None:
            print("Warning: label_map not provided. Falling back to numeric class indices.")
            label_map = {}

    # Load all adversarial samples
    adv_loader = DataLoader(adv_dataset, batch_size=len(adv_dataset), shuffle=False)
    images, labels = next(iter(adv_loader))

    # Send to model device
    device = next(model.parameters()).device
    images = images.to(device)

    # Run inference
    model.eval()
    with torch.no_grad():
        preds = model(images)
        pred_labels = preds.argmax(dim=1).cpu()

    # Plotting
    num_images = min(len(images), max_images)
    fig = plt.figure(figsize=(10, math.ceil(num_images / 6)))

    for i in range(num_images):
        ax = fig.add_subplot(math.ceil(num_images / 6), 6, i + 1)
        img = images[i].cpu()

        # Reshape for visualization
        if img.ndim == 3 and img.shape[0] == 1:  # (1, H, W)
            img = img.squeeze(0)
        elif img.ndim == 3 and img.shape[0] == 3:  # (3, H, W)
            img = img.permute(1, 2, 0)

        # De-normalize if image is in [-1, 1]
        if dataset_name in ["TinyImageNet", "TrafficSigns", "MNIST", "EMNIST", "SVHN", "USPS"]:
            img = (img + 1.0) / 2.0
            img = img.clamp(0, 1)

        # Label formatting
        orig = labels[i].item()
        pred = pred_labels[i].item()

        orig_name = label_map.get(orig, str(orig)) if label_map else str(orig)
        pred_name = label_map.get(pred, str(pred)) if label_map else str(pred)

        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(f"{orig_name} → {pred_name}", fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# Only plot adversarial images if they were actually generated (adversarial_training only)
if defense_type == "adversarial_training":
    if dataset_name == 'ReflectSign':
        plot_adversarial_images(
            adv_dataset,
            model=clean_model,
            dataset_name=dataset_name,
            label_map=idx_to_class,
            max_images=10
        )
    else:
        plot_adversarial_images(
            adv_dataset,
            model=clean_model,
            dataset_name=dataset_name,
            label_map=label_map,
            max_images=10
        )

    # Additional processing for adversarial images
    from torch.utils.data import DataLoader, TensorDataset

    # Step 1: Safety check
    if adv_dataset is None or len(adv_dataset) == 0:
        raise ValueError("Adversarial dataset is empty. Cannot proceed with training.")

    # Step 2: Load full adversarial dataset into memory
    device_type = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    adv_loader = DataLoader(
        adv_dataset,
        batch_size=len(adv_dataset),
        shuffle=False,
        num_workers=1,
        pin_memory=True if device_type.type == 'cuda' else False
    )

    # Step 3: Extract all tensors
    adv_images, adv_labels = next(iter(adv_loader))

    # Step 4: Rewrap as TensorDataset
    adv_ds = TensorDataset(adv_images, adv_labels)

    # Step 5: Batch size and batch count
    adv_batch_size = 128
    num_adv_images = len(adv_ds)
    num_adv_batches = (num_adv_images + adv_batch_size - 1) // adv_batch_size

    # Step 6: Confirm
    print(f"Number of adversarial images: {num_adv_images}")
    print(f"Number of batches (with batch size {adv_batch_size}): {num_adv_batches}")

from torch.utils.data import DataLoader, TensorDataset, Subset
import torch
from PIL import Image

# Wrapper dataset that handles corrupted images
class SafeDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        try:
            return self.dataset[idx]
        except Exception as e:
            #print(f"Skipping corrupted image at index {idx}: {e}")
            return None


# OPTIMIZATION: Reuse already-filtered datasets instead of filtering again
# filtered_train_set and filtered_test_set were already created earlier
# No need to call filtered_dataset() again - this was causing slow double-filtering
print(f"\n✓ Reusing filtered datasets (already created during initial setup)")
print(f"  Filtered training samples: {len(filtered_train_set)}")
print(f"  Filtered test samples: {len(filtered_test_set)}")

# TinyImageNet: Update label_map AFTER filtering
if dataset_name == 'TinyImageNet' and tinyimagenet_selected_classes is not None:
    print(f"\n{'='*80}")
    print(f"UPDATING LABEL_MAP FOR FILTERED TINYIMAGENET CLASSES")
    print(f"{'='*80}")

    # Load words.txt to get human-readable names
    words_txt_path = '/content/tiny-imagenet-200/words.txt'
    if os.path.exists(words_txt_path):
        wnid_to_words = {}
        with open(words_txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    wnid_to_words[parts[0]] = parts[1].split(',')[0].strip()

        # Get base classes
        if hasattr(train_set, 'classes'):
            base_classes = train_set.classes
        elif hasattr(train_set, 'dataset') and hasattr(train_set.dataset, 'classes'):
            base_classes = train_set.dataset.classes
        else:
            base_classes = None

        if base_classes:
            # Create idx_to_class for ALL classes
            idx_to_class_full = {i: wnid_to_words.get(wnid, wnid) for i, wnid in enumerate(base_classes)}

            # Now create label_map for FILTERED classes only
            label_map = {}
            for new_idx, orig_idx in enumerate(tinyimagenet_selected_classes):
                class_name = idx_to_class_full.get(orig_idx, f"Class_{orig_idx}")
                label_map[new_idx] = class_name

            # CRITICAL FIX: Also update idx_to_class here to match the new label_map
            idx_to_class = label_map

            print(f"[OK] Created label_map for filtered TinyImageNet classes:")
            for new_idx, name in label_map.items():
                orig_idx = tinyimagenet_selected_classes[new_idx]
                wnid = base_classes[orig_idx] if orig_idx < len(base_classes) else "unknown"
                print(f"  Label {new_idx}: {name} (original class {orig_idx}, wnid: {wnid})")
            print(f"{'='*80}\n")
        else:
            print("WARNING: Could not get base_classes for TinyImageNet")
    else:
        print(f"WARNING: words.txt not found at {words_txt_path}")

# Step 2: Ensure non-empty dataset
if len(filtered_train_set) == 0:
    raise ValueError("Filtered training set is empty. Check your keep_classes.")

# Wrap with SafeDataset to handle corrupted images
safe_train_set = SafeDataset(filtered_train_set)

# Step 3: Load full dataset to memory with error handling
device_type = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clean_loader = DataLoader(
    safe_train_set,
    batch_size=len(safe_train_set),
    shuffle=False,
    num_workers=0,
    pin_memory=True if device_type.type == 'cuda' else False
)

# Step 4: Extract all tensors (filter out None values)
batch = next(iter(clean_loader))
if batch is None:
    raise ValueError("All images are corrupted!")

clean_images, clean_labels = batch

# Step 5: Wrap as a TensorDataset
clean_ds = TensorDataset(clean_images, clean_labels)

# Step 6: Compute batches
clean_batch_size = 128
num_clean_images = len(clean_ds)
num_clean_batches = (num_clean_images + clean_batch_size - 1) // clean_batch_size

# Step 7: Print info
print("Number of clean images in training set:", num_clean_images)
print(f"Number of batches (batch size {clean_batch_size}): {num_clean_batches}")

from torch.utils.data import DataLoader, TensorDataset

# For randomization and input_transformation: Skip combining, use clean_loader directly
if defense_type in ["randomization", "input_transformation"]:
    print(f"\n{defense_type.capitalize()} defense: Using clean dataset only (no adversarial images needed)")
    num_adv_images = 0
    num_combined_images = num_clean_images
    num_combined_batches = num_clean_batches
    combineloader = clean_loader  # Use clean_loader directly

else:
    # For adversarial_training ONLY: Combine datasets
    # Shape and label adjustment based on dataset
    if dataset_name in ["MNIST", "EMNIST", "USPS"]:
        if adv_images.ndim == 3:
            adv_images = adv_images.unsqueeze(1)  # (N, H, W) → (N, 1, H, W)
        if clean_images.ndim == 3:
            clean_images = clean_images.unsqueeze(1)

    elif dataset_name == "Semeion":
        if adv_labels.ndim > 1:
            adv_labels = adv_labels.argmax(dim=1)
        if clean_labels.ndim > 1:
            clean_labels = clean_labels.argmax(dim=1)

    # Combine datasets (cast to consistent dtype if needed)
    combined_images = torch.cat((adv_images, clean_images)).float()
    combined_labels = torch.cat((adv_labels, clean_labels)).long()
    combined_tensor_dataset = TensorDataset(combined_images, combined_labels)

    # DataLoader config
    combined_batch_size = 128
    num_combined_images = len(combined_tensor_dataset)
    num_combined_batches = (num_combined_images + combined_batch_size - 1) // combined_batch_size

    combineloader = DataLoader(
        combined_tensor_dataset,
        batch_size=combined_batch_size,
        shuffle=True,
        num_workers=1,
        pin_memory=True if torch.cuda.is_available() else False
    )

    # Log stats
    print(f"Total combined images: {num_combined_images}")
    print(f"Total batches (batch size {combined_batch_size}): {num_combined_batches}")
    print(f"Number of combined images: {num_combined_images}")
    print(f"Number of batches in combined loader: {num_combined_batches}")

"""Randomization defense techniques involve applying random transformations to the input data during training to increase the model's robustness against adversarial attacks. The key idea behind randomization defense is to introduce random variations in the input data, making it harder for adversarial perturbations to have a consistent effect on the model's predictions.
The implemented randomization defense methods work as follows:

Random Resizing: The input images are randomly resized within a specified scale range. This introduces variations in the spatial dimensions of the images, making the model more resilient to size-related adversarial perturbations.

Random Cropping: A random smaller region is cropped from the input images. This helps the model learn to focus on different parts of the image and reduces its sensitivity to specific pixel locations.
Random Rotation: The input images are randomly rotated within a specified angle range. This helps the model become invariant to rotational changes and enhances its ability to recognize objects from different orientations.

Combined Randomization: Multiple randomization techniques, including resizing, cropping, rotation, color jittering, random erasing, and noise injection, are applied together. This creates a diverse set of input variations, making it challenging for adversarial perturbations to have a consistent impact.
"""

import torch
from torchvision.transforms import ToTensor

class TransformedDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform if transform is not None else ToTensor()

    def __getitem__(self, index):
        x, y = self.dataset[index]

        # Convert non-tensor input to tensor (if needed)
        if not isinstance(x, torch.Tensor):
            if hasattr(x, "convert"):  # PIL Image
                x = x.convert("RGB") if x.mode != "RGB" else x
            x = self.transform(x)  # Always apply transform after conversion
        else:
            x = x.float()

        # If x is still 2D (grayscale), unsqueeze to add channel
        if x.ndim == 2:
            x = x.unsqueeze(0)

        return x, y

    def __len__(self):
        return len(self.dataset)

def save_retrained_model(model, save_dir):

    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    model_path = f"{save_dir}/retrained_model.pth"

    torch.save(model.state_dict(), model_path)
    #print(f"Saved retrained_model state dict to {model_path}")

"""Input Transformation
Input transformation is a defense technique that aims to preprocess the input data before feeding it into the model. The goal is to make the input more robust and less susceptible to adversarial perturbations. This project implements three input transformation approaches:

Image Quilting: This technique involves dividing the input image into smaller patches and randomly replacing some of the patches with patches from clean images. The idea is to introduce randomness and break up the adversarial patterns.

Adversarial Logit Pairing: In this approach, adversarial examples are generated using the Fast Gradient Sign Method (FGSM), and the model is trained to minimize the difference between the logits (pre-softmax outputs) of clean examples and their corresponding adversarial examples. This encourages the model to have similar predictions for clean and adversarial inputs.

Differential Privacy: Differential privacy is a technique that adds controlled noise to the input data to protect the privacy of individual examples. In the context of adversarial defense, adding noise can help mitigate the impact of adversarial perturbations by making them less effective.

"""

import numpy as np
import torch
from skimage.util import view_as_windows
from torchvision.transforms import ToTensor

def reconstruct_image(patches, patch_size, image_shape):
    """
    Reconstruct a single image from non-overlapping patches.
    """
    h, w = image_shape
    rows = h // patch_size
    cols = w // patch_size
    reconstructed = np.zeros((h, w), dtype=patches.dtype)

    idx = 0
    for i in range(rows):
        for j in range(cols):
            reconstructed[i * patch_size:(i + 1) * patch_size,
                          j * patch_size:(j + 1) * patch_size] = patches[idx]
            idx += 1

    return reconstructed

def apply_image_quilting(images, patch_size=4):
    """
    Apply non-overlapping image quilting to grayscale or RGB image(s).
    Supports both single image (C, H, W) and batch (N, C, H, W).
    CRITICAL: Preserves exact input dimensions and batch size.

    Args:
        images: torch.Tensor or PIL.Image
        patch_size: Size of square patch
    Returns:
        torch.Tensor of EXACT same shape as input
    """
    # Ensure input is a tensor (handle PIL.Image)
    if not isinstance(images, torch.Tensor):
        images = ToTensor()(images)

    single_image = False
    if images.dim() == 3:
        images = images.unsqueeze(0)
        single_image = True
    elif images.dim() != 4:
        raise ValueError("Input tensor must have shape (N, C, H, W) or (C, H, W)")

    # CRITICAL: Store original for fallback
    original_images = images.clone()
    original_shape = images.shape
    N, C, H, W = images.shape

    # Check if image is patch-aligned
    if H % patch_size != 0 or W % patch_size != 0:
        print(f"WARNING: Image size ({H}x{W}) not divisible by patch_size ({patch_size})")
        print(f"Skipping image quilting, returning original images")
        return original_images.squeeze(0) if single_image else original_images

    try:
        # Convert to NumPy (keep full size)
        images_np = images.detach().cpu().numpy()
        quilted_np = np.zeros_like(images_np)  # Use zeros instead of empty for safety

        # Process each image in batch
        for i in range(N):
            for c in range(C):
                channel = images_np[i, c]

                # Extract non-overlapping patches
                patches = view_as_windows(channel, (patch_size, patch_size), step=patch_size)
                patches = patches.reshape(-1, patch_size, patch_size)

                # Random shuffle (with replacement to simulate quilting)
                shuffled = patches[np.random.choice(len(patches), len(patches), replace=True)]

                # Reconstruct the image
                quilted_np[i, c] = reconstruct_image(shuffled, patch_size, (H, W))

        # Convert back to tensor
        output = torch.from_numpy(quilted_np).to(images.device).to(images.dtype)

        # CRITICAL: Verify shape is EXACTLY preserved
        if output.shape != original_shape:
            print(f"ERROR: Image quilting changed shape: {original_shape} -> {output.shape}")
            print(f"Returning original images")
            return original_images.squeeze(0) if single_image else original_images

        return output.squeeze(0) if single_image else output

    except Exception as e:
        print(f"ERROR: Image quilting failed: {e}")
        print(f"Returning original images with shape {original_shape}")
        return original_images.squeeze(0) if single_image else original_images

import torch
import torch.nn.functional as F

def apply_adversarial_logit_pairing(model, images, labels=None, epsilon=0.1, clamp_min=0.0, clamp_max=1.0):
    """
    Apply Adversarial Logit Pairing (ALP) to input images.

    Args:
        model: PyTorch model
        images: Input images (B, C, H, W)
        labels: Optional ground-truth labels (B,)
        epsilon: Perturbation strength
        clamp_min: Minimum pixel value (e.g., 0.0)
        clamp_max: Maximum pixel value (e.g., 1.0)

    Returns:
        adversarial_images: Perturbed adversarial samples
        loss_pairing: MSE loss between clean and adversarial logits
    """
    device = next(model.parameters()).device
    images = images.to(device)
    images.requires_grad = False

    if labels is None:
        with torch.no_grad():
            labels = model(images).argmax(dim=1)
    labels = labels.to(device)

    # Step 1: Generate adversarial images
    adv_input = images.clone().detach().requires_grad_()
    logits = model(adv_input)
    loss = F.cross_entropy(logits, labels)
    model.zero_grad()
    loss.backward()
    grad = adv_input.grad.detach()
    adversarial_images = adv_input + epsilon * grad.sign()
    adversarial_images = torch.clamp(adversarial_images, clamp_min, clamp_max)

    # Step 2: Logit pairing loss (used later during training)
    with torch.no_grad():
        logits_clean = model(images)
    logits_adv = model(adversarial_images)
    loss_pairing = F.mse_loss(logits_adv, logits_clean)

    return adversarial_images.detach(), loss_pairing.detach()

import torch
import numpy as np

def apply_gaussian_noise_defense(images, epsilon=1.0, sensitivity=1.0, clamp_min=0.0, clamp_max=1.0):
    """
    Adds Gaussian noise to images as adversarial defense.

    Args:
        images (Tensor): Input tensor of shape (N, C, H, W)
        epsilon (float): Privacy budget (smaller = more noise)
        sensitivity (float): Sensitivity of the query (often 1.0)
        clamp_min (float): Min pixel value (0 or -1)
        clamp_max (float): Max pixel value (1)

    Returns:
        noisy_images (Tensor): Images with added Gaussian noise
    """
    original_batch_size = images.size(0) if images.dim() == 4 else 1
    device = images.device
    delta = 1e-2  # Set default δ value
    scale = sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    noise = torch.normal(mean=0, std=scale, size=images.shape).to(device)
    noisy_images = images + noise
    noisy_images = torch.clamp(noisy_images, clamp_min, clamp_max)

    # CRITICAL: Verify batch size is preserved
    result_batch_size = noisy_images.size(0) if noisy_images.dim() == 4 else 1
    assert result_batch_size == original_batch_size, f"Batch size changed: {original_batch_size} -> {result_batch_size}"

    return noisy_images

def apply_combined_input_transformation(model, images, patch_size=4, epsilon_alp=0.1, epsilon_dp=1.0, clamp_min=0.0, clamp_max=1.0):
    """
    Applies a sequence of transformations: image quilting → JPEG compression → Gaussian noise.
    Args:
        model: The neural network model (not used currently).
        images (Tensor): Input batch of shape (N, C, H, W).
        patch_size (int): Size for quilting.
        epsilon_alp (float): Unused (legacy parameter).
        epsilon_dp (float): Noise scale parameter.
        clamp_min (float): Min pixel value after noise.
        clamp_max (float): Max pixel value after noise.
    Returns:
        Tensor: Transformed images.
    """
    original_batch_size = images.size(0) if images.dim() == 4 else 1

    # Step 1: Quilting
    quilted_images = apply_image_quilting(images, patch_size)

    # Step 2: JPEG Compression
    compressed_images = apply_jpeg_compression(quilted_images, quality=75)

    # Step 3: Gaussian Noise
    transformed_images = apply_gaussian_noise_defense(compressed_images, epsilon=epsilon_dp, clamp_min=clamp_min, clamp_max=clamp_max)

    # CRITICAL: Verify batch size is preserved
    result_batch_size = transformed_images.size(0) if transformed_images.dim() == 4 else 1
    assert result_batch_size == original_batch_size, f"Batch size changed: {original_batch_size} -> {result_batch_size}"

    return transformed_images

def apply_jpeg_compression(images, quality=75):
    """
    Apply JPEG compression as input transformation.
    Args:
        images: Input tensor (B, C, H, W) or (C, H, W)
        quality: JPEG quality (1-100, lower = more compression)
    Returns:
        Compressed images as tensor
    """
    import io
    from PIL import Image
    import torchvision.transforms as T

    single_image = False
    if images.dim() == 3:
        images = images.unsqueeze(0)
        single_image = True

    original_batch_size = images.size(0)
    batch_size = images.size(0)
    compressed_images = []

    to_pil = T.ToPILImage()
    to_tensor = T.ToTensor()

    for i in range(batch_size):
        try:
            img = to_pil(images[i].cpu())
            buffer = io.BytesIO()
            img.save(buffer, format='JPEG', quality=quality)
            buffer.seek(0)
            compressed_img = Image.open(buffer)
            compressed_images.append(to_tensor(compressed_img))
        except Exception as e:
            print(f"WARNING: JPEG compression failed for image {i}: {e}")
            compressed_images.append(images[i].cpu())

    result = torch.stack(compressed_images).to(images.device)

    # CRITICAL: Verify batch size is preserved
    if result.size(0) != original_batch_size:
        print(f"ERROR: JPEG compression changed batch size: {original_batch_size} -> {result.size(0)}")
        return images

    return result.squeeze(0) if single_image else result

def apply_bit_depth_reduction(images, bits=4):
    """
    Reduce bit depth of images (quantization).
    Args:
        images: Input tensor (B, C, H, W) or (C, H, W)
        bits: Number of bits (1-8, lower = more reduction)
    Returns:
        Quantized images
    """
    original_batch_size = images.size(0) if images.dim() == 4 else 1
    levels = 2 ** bits
    quantized = torch.round(images * (levels - 1)) / (levels - 1)
    result = torch.clamp(quantized, 0.0, 1.0)

    # CRITICAL: Verify batch size is preserved
    result_batch_size = result.size(0) if result.dim() == 4 else 1
    assert result_batch_size == original_batch_size, f"Batch size changed: {original_batch_size} -> {result_batch_size}"

    return result

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset
import torchvision.transforms as transforms
import multiprocessing
import time
import os
import matplotlib.pyplot as plt

# --- Helper: print images with word labels ---
def print_images_with_labels(images, labels, outputs=None, n=3, dataset_name=None, label_map=None):
    def denormalize(img_tensor):
        return (img_tensor * 0.5) + 0.5  # Assumes Normalize((0.5,), (0.5,))

    images = images[:n]
    labels = labels[:n]
    if outputs is not None:
        outputs = outputs[:n]

    fig = plt.figure(figsize=(10, 2))
    for i in range(n):
        ax = fig.add_subplot(1, n, i + 1)

        img = images[i]
        if img.ndim == 3 and img.shape[0] == 1:  # Grayscale
            img = img.squeeze(0)
            img = denormalize(img)
        elif img.ndim == 3 and img.shape[0] == 3:  # RGB
            img = img.permute(1, 2, 0)
            img = denormalize(img)
        else:
            img = denormalize(img)

        img = img.clamp(0, 1)  #  Ensure final range is [0, 1]

        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)

        orig_label = labels[i].item()
        pred_label = outputs[i].argmax().item() if outputs is not None else None

        orig_name = label_map.get(orig_label, str(orig_label)) if label_map else str(orig_label)
        pred_name = label_map.get(pred_label, str(pred_label)) if label_map and pred_label is not None else str(pred_label)

        ax.set_title(f"{orig_name} → {pred_name}", fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# --- Randomization Defense Support ---
class RandomizedDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.base_dataset[index]
        return self.transform(x), y

    def __len__(self):
        return len(self.base_dataset)

def add_random_noise(x):
    return x + 0.05 * torch.randn_like(x)

def apply_randomization_defense(dataset, randomization_defense):
    if randomization_defense == "random_resizing":
        transform = transforms.Compose([
            transforms.RandomResizedCrop(size=(28, 28), scale=(0.8, 1.2)),  # Match training
            transforms.ToTensor()
        ])
    elif randomization_defense == "random_cropping":
        transform = transforms.Compose([
            transforms.RandomCrop(28, padding=4),  # MATCH TRAINING EXACTLY
            transforms.ToTensor()
        ])
    elif randomization_defense == "random_rotation":
        transform = transforms.Compose([
            transforms.RandomRotation(degrees=15),  # Match training
            transforms.ToTensor()
        ])
    elif randomization_defense == "combined_randomization":
        transform = transforms.Compose([
            transforms.RandomCrop(28, padding=4),  # Match training
            transforms.RandomRotation(degrees=15),  # Match training
            transforms.ToTensor()
        ])
    else:
        transform = transforms.ToTensor()

    dataset.transform = transform
    return dataset

### Fixed
def adversarial_retraining(
    model,
    combined_loader,
    optimizer,
    loss_fn,
    num_epochs,
    device,
    defense_type,
    randomization_defense=None,
    input_transformation=None,
    clean_loader=None,
    batch_size=128,
    learning_rate=0.001,
    weight_decay=0.0001,
    print_images_per_epoch=1,
    save_dir="./retrained_model",
    dataset_name=None,
    label_map=None,
    attack=None  # ADDED: Need attack for randomization defense
):
    """
    Retrain model with defense mechanism following dissertation procedures:
    - Figure 4.11 (Adversarial Training - 16 steps)
    - Figure 4.10 (Randomization - 17 steps)
    - Figure 4.9 (Input Transformation - 13 steps)
    - Table 4.4 (Dataset specifications)
    """
    os.makedirs(save_dir, exist_ok=True)
    model.to(device)

    for param_group in optimizer.param_groups:
        param_group['lr'] = learning_rate
        param_group['weight_decay'] = weight_decay

    train_losses = []
    train_accuracies = []


    # ============================================================================
    # RANDOMIZATION DEFENSE - 17 Steps (Figure 4.10)
    # ============================================================================
    if defense_type == "randomization":
      print(f"\n{'='*80}")
      print(f"RANDOMIZATION DEFENSE: {randomization_defense}")
      print(f"{'='*80}\n")

      print("Standard randomization defense: NO RETRAINING")
      print("Randomization will be applied at TEST time during evaluation")

      # Save clean model as final_model.pth for consistency
      model_path = os.path.join(save_dir, "final_model.pth")
      torch.save(model.state_dict(), model_path)
      print(f" Clean model saved to: {model_path}\n")

      return TensorDataset(torch.empty(0), torch.empty(0, dtype=torch.long))

    # ============================================================================
    # INPUT TRANSFORMATION DEFENSE - 13 Steps (Figure 4.9)
    # ============================================================================
    # ============================================================================
    # INPUT TRANSFORMATION DEFENSE - Apply transforms at TEST TIME only
    # ============================================================================
    elif defense_type == "input_transformation":
        print(f"\n{'='*80}")
        print(f"INPUT TRANSFORMATION DEFENSE: {input_transformation}")
        print(f"{'='*80}\n")

        print("INPUT TRANSFORMATION: NO RETRAINING")
        print("Transformations will be applied at TEST TIME during evaluation")

        # Save clean model as final_model.pth for consistency
        model_path = os.path.join(save_dir, "final_model.pth")
        torch.save(model.state_dict(), model_path)
        print(f" Clean model saved to: {model_path}\n")

        return TensorDataset(torch.empty(0), torch.empty(0, dtype=torch.long))

    # ============================================================================
    # ADVERSARIAL TRAINING DEFENSE - 16 Steps (Figure 4.11)
    # ============================================================================
    elif defense_type == "adversarial_training":
        print(f"\n{'='*80}")
        print(f"ADVERSARIAL TRAINING DEFENSE")
        print(f"{'='*80}\n")

        # Steps 1-10: Already done (combined_loader passed in with clean + adversarial)
        # Per Table 4.4: Final Dataset = "Combined Normal Sample and Adversarial Sample"
        print("Using pre-combined dataset (clean + adversarial)")
        print(f"  Total samples: {len(combined_loader.dataset)}")

        # STEP 11: RETRAIN happens in main loop below

    else:
        raise ValueError(f"Unsupported defense_type: {defense_type}")

    # ============================================================================
    # MAIN TRAINING LOOP (Steps 7 or 11 depending on defense)
    # ============================================================================
    print("\n⏳ Training/Retraining in progress...")

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        retrained_imgs = []
        retrained_labels = []

        for i, (imgs, labels) in enumerate(combined_loader):
            imgs, labels = imgs.to(device), labels.to(device)
            output = model(imgs)
            loss = loss_fn(output, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = torch.max(output, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

            if i < print_images_per_epoch:
                print_images_with_labels(
                    imgs.cpu(),
                    labels.cpu(),
                    output.cpu(),
                    n=3,
                    dataset_name=dataset_name,
                    label_map=label_map
                )

            retrained_imgs.append(imgs.cpu())
            retrained_labels.append(labels.cpu())

        avg_loss = train_loss / len(combined_loader)
        accuracy = correct / total
        train_losses.append(avg_loss)
        train_accuracies.append(accuracy)

        print(f"Train Loss: {avg_loss:.4f}, Train Accuracy: {accuracy:.2%}")

    # Save the DEFENSE-TRAINED model (not the clean model!)
    model_path = os.path.join(save_dir, "final_model.pth")
    torch.save(model.state_dict(), model_path)
    print(f"\n Defense-trained model saved to: {model_path}")
    print(f"   (This is the model that should be loaded for evaluation)\n")

    retrained_dataset = TensorDataset(torch.cat(retrained_imgs), torch.cat(retrained_labels))
    return retrained_dataset

import time
from tqdm import tqdm

# Display configuration
print(f"\nStarting defense training using defense type: {defense_type}")
if defense_type == "randomization":
    print(f"Randomization technique: {randomization_defense}")
elif defense_type == "input_transformation":
    print(f"Input transformation technique: {input_transformation}")
elif defense_type == "adversarial_training":
    print(f"Using combined adversarial + clean dataset")
else:
    raise ValueError(f"Unknown defense type: {defense_type}")

# Start the timer
start_time_retrain = time.time()

# Start training
print("\nTraining in progress...")

# CRITICAL FIX: Save original clean model BEFORE adversarial training
# This prevents the PRE-ATTACK evaluation from using the trained model
import copy
original_clean_model = copy.deepcopy(clean_model)
original_clean_model.eval()
print(f"[OK] Original clean model saved for PRE-ATTACK evaluation\n")

# Use clean_model directly for ALL defenses (same as MNIST/EMNIST)
adv_trained_dataset = adversarial_retraining(
    model=clean_model,  # This will be modified by adversarial training
    combined_loader=combineloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    num_epochs=num_epochs,
    device=device,
    defense_type=defense_type,
    randomization_defense=randomization_defense,
    input_transformation=input_transformation,
    clean_loader=clean_loader,
    save_dir="./retrained_model",
    dataset_name=dataset_name,
    label_map=label_map if dataset_name != 'ReflectSign' else idx_to_class,
    attack=attack
)


# End the timer
end_time_retrain = time.time()
elapsed = end_time_retrain - start_time_retrain

# Summary
minutes = int(elapsed // 60)
seconds = int(elapsed % 60)
print(f"\nRetraining complete in {minutes}m {seconds}s")
print(f"Total samples in retrained dataset: {len(adv_trained_dataset)}")

# Calculate training duration
retraining_duration_seconds = end_time_retrain - start_time_retrain
retraining_duration_minutes = retraining_duration_seconds / 60
retraining_duration_hours = retraining_duration_minutes / 60

# Print training duration in multiple units
print(f"Training duration: {retraining_duration_seconds:.2f} seconds")
print(f"Training duration: {retraining_duration_minutes:.2f} minutes")
print(f"Training duration: {retraining_duration_hours:.2f} hours")

# Optional: friendly HH:MM:SS format
h, rem = divmod(retraining_duration_seconds, 3600)
m, s = divmod(rem, 60)
print(f"Elapsed time (formatted): {int(h)}h {int(m)}m {int(s)}s")

num_clean_elements = 0
num_adv_elements = 0

# Count total clean images
for clean_images, _ in clean_loader:
    num_clean_elements += len(clean_images)
print(f"Number of clean images: {num_clean_elements}")

# Count total adversarial images
for adv_images, _ in adv_loader:
    num_adv_elements += len(adv_images)
print(f"Number of adversarial images: {num_adv_elements}")

# Total combined
num_combined_elements = num_clean_elements + num_adv_elements
print(f"Number of combined images: {num_combined_elements}")

# Count adversarially trained dataset elements
if hasattr(adv_trained_dataset, "tensors"):
    adv_trained_images_tensor, _ = adv_trained_dataset.tensors
    num_adv_trained_elements = len(adv_trained_images_tensor)
else:
    num_adv_trained_elements = len(adv_trained_dataset)
print(f"Number of adversarially trained images: {num_adv_trained_elements}")

import os
import torch

def load_retrained_model(model, save_dir, filename="clean_model.pth", device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model_path = os.path.join(save_dir, filename)

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found at: {model_path}")

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    print(f"Loaded model from: {model_path}")
    return model

import torch

def evaluate_after_adv_retraining(model, data_loader, loss_fn, device, label_map=None, dataset_name=None):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    misclassified_images = []
    misclassified_labels = []
    misclassified_preds = []

    dataset_size = len(data_loader.dataset)
    print(f"Number of test images: {dataset_size}")

    with torch.no_grad():
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * x.size(0)

            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == y).sum().item()
            total_samples += x.size(0)

            incorrect = preds != y
            if incorrect.any():
                x_cpu = x[incorrect].cpu()
                y_cpu = y[incorrect].cpu()
                preds_cpu = preds[incorrect].cpu()

                misclassified_images.append(x_cpu)
                misclassified_labels.append(y_cpu)
                misclassified_preds.append(preds_cpu)

                # Decode class indices to human-readable labels if label_map is provided
                if label_map:
                    decoded_true = [label_map[int(label)] for label in y_cpu]
                    decoded_pred = [label_map[int(pred)] for pred in preds_cpu]

                    print(f"\n[{dataset_name or 'Dataset'}] Misclassified examples:")
                    for true_lbl, pred_lbl in zip(decoded_true, decoded_pred):
                        print(f"  🔻 True: {true_lbl} | 🔺 Predicted: {pred_lbl}")

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    num_misclassified = total_samples - total_correct

    print(f"\nNumber of misclassified images: {num_misclassified} out of {total_samples}")
    print(f"Defense Test loss: {avg_loss:.4f}, Accuracy: {accuracy:.2%}")

    if not misclassified_images:
        return avg_loss, accuracy, torch.empty(0), torch.empty(0), torch.empty(0)

    return (
        avg_loss,
        accuracy,
        torch.cat(misclassified_images),
        torch.cat(misclassified_labels),
        torch.cat(misclassified_preds),
    )

# ============================================================================
# PHASE 4: EVALUATE DEFENSE MODEL (CORRECTED)
# ============================================================================
print(f"\n{'='*100}")
print("PHASE 4: EVALUATING DEFENSE-TRAINED MODEL")
print(f"{'='*100}\n")

# Load the defense model (the retrained model with defense)
save_dir = "./retrained_model"
defense_model = load_retrained_model(model, save_dir, filename="final_model.pth")

# ============================================================================
# PRE-ATTACK EVALUATION (CLEAN MODEL on clean test data)
# ============================================================================
print(f"{'='*80}")
print("PRE-ATTACK EVALUATION (ORIGINAL CLEAN MODEL on Clean Test Data)")
print(f"{'='*80}\n")

# Use the ORIGINAL clean model (before adversarial training)
original_clean_model.eval()
total_correct = 0
total_samples = 0
total_loss = 0.0
pre_attack_misclassified_images = []
pre_attack_misclassified_labels = []
pre_attack_misclassified_preds = []

with torch.no_grad():
    for imgs, labels in filtered_test_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        # NO TRANSFORMS - just evaluate ORIGINAL CLEAN MODEL on clean data
        outputs = original_clean_model(imgs)
        loss = loss_fn(outputs, labels)

        total_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        # Track misclassifications
        incorrect = preds != labels
        if incorrect.any():
            pre_attack_misclassified_images.append(imgs[incorrect].cpu())
            pre_attack_misclassified_labels.append(labels[incorrect].cpu())
            pre_attack_misclassified_preds.append(preds[incorrect].cpu())

pre_attack_test_acc = total_correct / total_samples
pre_attack_test_loss = total_loss / total_samples

# Concatenate misclassifications
if pre_attack_misclassified_images:
    pre_attack_misclassified_images = torch.cat(pre_attack_misclassified_images)
    pre_attack_misclassified_labels = torch.cat(pre_attack_misclassified_labels)
    pre_attack_misclassified_preds = torch.cat(pre_attack_misclassified_preds)
else:
    pre_attack_misclassified_images = torch.empty(0)
    pre_attack_misclassified_labels = torch.empty(0)
    pre_attack_misclassified_preds = torch.empty(0)

print(f" PRE-ATTACK ACCURACY: {pre_attack_test_acc:.2%}")
print(f" PRE-ATTACK LOSS: {pre_attack_test_loss:.4f}")
print(f"   (This is the ORIGINAL CLEAN MODEL's accuracy BEFORE adversarial training)\n")

# ============================================================================
# REUSE ADVERSARIAL TEST DATA (Don't generate again!)
# ============================================================================
print(f"{'='*80}")
print("USING EXISTING ADVERSARIAL TEST DATA")
print(f"{'='*80}\n")

# Reuse the adversarial test data that was already generated for clean model evaluation
# This ensures fair comparison: both models tested on SAME adversarial images
print(f"Reusing adversarial test data generated from attacking clean model")
print(f"Total adversarial test samples: {len(adv_attack_loader.dataset)}\n")

attacked_test_loader = adv_attack_loader  # Just reuse it!

# ============================================================================
# POST-ATTACK EVALUATION (Defense model on attacked test data)
# ============================================================================

print(f"{'='*80}")
print("POST-ATTACK EVALUATION (Defense Model on Attacked Test Data)")
print(f"{'='*80}\n")

defense_model.eval()
total_correct = 0
total_samples = 0
total_loss = 0.0
retrained_misclassified_images = []
retrained_misclassified_labels = []
retrained_misclassified_preds = []

with torch.no_grad():
    for batch_idx, (imgs, labels) in enumerate(attacked_test_loader):
        imgs, labels = imgs.to(device), labels.to(device)

        # DEBUG: Check initial batch size
        initial_batch_size = imgs.size(0)
        initial_label_size = labels.size(0)
        if initial_batch_size != initial_label_size:
            print(f"ERROR in batch {batch_idx}: Initial mismatch - imgs={initial_batch_size}, labels={initial_label_size}")

        if defense_type == "randomization":
            # OPTIMIZED: Use batch GPU operations
            if randomization_defense == "random_resizing":
                # Apply dataset-specific random resizing
                if dataset_name in ['TinyImageNet', 'TrafficSigns', 'ReflectSign']:
                    # For 64×64 RGB images
                    scale = torch.FloatTensor(1).uniform_(0.8, 1.2).item()
                    new_size = int(imgs.shape[2] * scale)
                    imgs = F.interpolate(imgs, size=(new_size, new_size), mode='bilinear', align_corners=False)
                    imgs = F.interpolate(imgs, size=(64, 64), mode='bilinear', align_corners=False)
                else:
                    # For 28×28 grayscale images
                    scale = torch.FloatTensor(1).uniform_(0.8, 1.2).item()
                    new_size = int(imgs.shape[2] * scale)
                    imgs = F.interpolate(imgs, size=(new_size, new_size), mode='bilinear', align_corners=False)
                    imgs = F.interpolate(imgs, size=(28, 28), mode='bilinear', align_corners=False)

            elif randomization_defense == "random_cropping":
                # Apply dataset-specific random cropping
                if dataset_name in ['TinyImageNet', 'TrafficSigns', 'ReflectSign']:
                    # For 64×64 RGB images: RandomCrop(64, padding=8)
                    # Pad by 8 pixels on each side (64->80), then crop 64x64
                    imgs = F.pad(imgs, (8, 8, 8, 8), mode='reflect')  # Now 80x80
                    i = torch.randint(0, 17, (1,)).item()  # Random offset 0-16 for top
                    j = torch.randint(0, 17, (1,)).item()  # Random offset 0-16 for left
                    imgs = imgs[:, :, i:i+64, j:j+64]  # Crop back to 64x64
                else:
                    # For 28×28 grayscale images: RandomCrop(28, padding=4)
                    # Pad by 4 pixels on each side (28->32), then crop 28x28
                    imgs = F.pad(imgs, (4, 4, 4, 4), mode='reflect')  # Now 32x32
                    i = torch.randint(0, 5, (1,)).item()  # Random offset 0-4 for top
                    j = torch.randint(0, 5, (1,)).item()  # Random offset 0-4 for left
                    imgs = imgs[:, :, i:i+28, j:j+28]  # Crop back to 28x28

            elif randomization_defense == "random_rotation":
                import math
                angle = torch.FloatTensor(1).uniform_(-15, 15).item()
                angle_rad = angle * 3.14159 / 180
                theta = torch.tensor([[math.cos(angle_rad), -math.sin(angle_rad), 0],
                                      [math.sin(angle_rad), math.cos(angle_rad), 0]],
                                     dtype=imgs.dtype, device=imgs.device).unsqueeze(0).repeat(imgs.size(0), 1, 1)
                grid = F.affine_grid(theta, imgs.size(), align_corners=False)
                imgs = F.grid_sample(imgs, grid, align_corners=False)

            elif randomization_defense == "combined_randomization":
                import math
                # Apply dataset-specific crop + rotation
                if dataset_name in ['TinyImageNet', 'TrafficSigns', 'ReflectSign']:
                    # For 64×64 RGB images: Crop with padding=8 + rotation
                    imgs = F.pad(imgs, (8, 8, 8, 8), mode='reflect')
                    i = torch.randint(0, 17, (1,)).item()
                    j = torch.randint(0, 17, (1,)).item()
                    imgs = imgs[:, :, i:i+64, j:j+64]
                else:
                    # For 28×28 grayscale images: Crop with padding=4 + rotation
                    imgs = F.pad(imgs, (4, 4, 4, 4), mode='reflect')
                    i = torch.randint(0, 5, (1,)).item()
                    j = torch.randint(0, 5, (1,)).item()
                    imgs = imgs[:, :, i:i+28, j:j+28]

                angle = torch.FloatTensor(1).uniform_(-15, 15).item()
                angle_rad = angle * 3.14159 / 180
                theta = torch.tensor([[math.cos(angle_rad), -math.sin(angle_rad), 0],
                                      [math.sin(angle_rad), math.cos(angle_rad), 0]],
                                     dtype=imgs.dtype, device=imgs.device).unsqueeze(0).repeat(imgs.size(0), 1, 1)
                grid = F.affine_grid(theta, imgs.size(), align_corners=False)
                imgs = F.grid_sample(imgs, grid, align_corners=False)

            # CRITICAL: Verify batch sizes match after randomization
            if imgs.size(0) != labels.size(0):
                print(f"ERROR in batch {batch_idx}: After randomization - imgs={imgs.size(0)}, labels={labels.size(0)}")
                raise ValueError(f"Batch size mismatch after randomization: imgs={imgs.size(0)}, labels={labels.size(0)}")

        elif defense_type == "input_transformation":
            # Apply same transformation as during training
            before_transform_size = imgs.size(0)

            if input_transformation == "image_quilting":
                imgs = apply_image_quilting(imgs)
            elif input_transformation == "jpeg_compression":
                imgs = apply_jpeg_compression(imgs, quality=75)
            elif input_transformation == "bit_depth_reduction":
                imgs = apply_bit_depth_reduction(imgs, bits=4)
            elif input_transformation == "gaussian_noise_defense":
                imgs = apply_gaussian_noise_defense(imgs)
            elif input_transformation == "combined_input_transformation":
                imgs = apply_combined_input_transformation(defense_model, imgs)

            # DEBUG: Check after transformation
            after_transform_size = imgs.size(0)
            if before_transform_size != after_transform_size:
                print(f"ERROR in batch {batch_idx}: {input_transformation} changed batch size {before_transform_size} -> {after_transform_size}")
                raise ValueError(f"Transformation {input_transformation} changed batch size: {before_transform_size} -> {after_transform_size}")

        # CRITICAL: Final verification before forward pass
        if imgs.size(0) != labels.size(0):
            print(f"FATAL ERROR in batch {batch_idx}: Before forward pass - imgs={imgs.size(0)}, labels={labels.size(0)}")
            print(f"imgs shape: {imgs.shape}, labels shape: {labels.shape}")
            print(f"Defense type: {defense_type}")
            if defense_type == "input_transformation":
                print(f"Input transformation: {input_transformation}")
            elif defense_type == "randomization":
                print(f"Randomization defense: {randomization_defense}")
            raise ValueError(f"Batch size mismatch before forward pass: imgs={imgs.size(0)}, labels={labels.size(0)}")

        outputs = defense_model(imgs)
        loss = loss_fn(outputs, labels)

        total_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        # Track misclassifications
        incorrect = preds != labels
        if incorrect.any():
            retrained_misclassified_images.append(imgs[incorrect].cpu())
            retrained_misclassified_labels.append(labels[incorrect].cpu())
            retrained_misclassified_preds.append(preds[incorrect].cpu())

post_attack_test_acc = total_correct / total_samples
retrained_test_loss = total_loss / total_samples

# Concatenate misclassifications
if retrained_misclassified_images:
    retrained_misclassified_images = torch.cat(retrained_misclassified_images)
    retrained_misclassified_labels = torch.cat(retrained_misclassified_labels)
    retrained_misclassified_preds = torch.cat(retrained_misclassified_preds)
else:
    retrained_misclassified_images = torch.empty(0)
    retrained_misclassified_labels = torch.empty(0)
    retrained_misclassified_preds = torch.empty(0)

print(f"POST-ATTACK ACCURACY: {post_attack_test_acc:.2%}")
print(f"   (This is the defense model's accuracy on attacked test data)\n")

# Set retrained_accuracy for backward compatibility with existing code
retrained_accuracy = post_attack_test_acc

if dataset_name == 'ReflectSign':
  retrained_misclassified_images, retrained_misclassified_labels, retrained_misclassified_preds = analyze_for_misclassifications(retrained_misclassified_images, retrained_misclassified_labels, retrained_misclassified_preds,label_map=idx_to_class)
else:
    retrained_misclassified_images, retrained_misclassified_labels, retrained_misclassified_preds = analyze_for_misclassifications(retrained_misclassified_images, retrained_misclassified_labels, retrained_misclassified_preds,label_map=label_map)

def precision(true_positives, false_positives):
    if true_positives + false_positives == 0:
        return 0.0
    return true_positives / (true_positives + false_positives)

def recall(true_positives, false_negatives):
    if true_positives + false_negatives == 0:
        return 0.0
    return true_positives / (true_positives + false_negatives)

def f1_score(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

from sklearn.metrics import roc_auc_score

def auc_score(true_labels, predicted_probabilities):
    if len(set(true_labels)) == 2:
        return roc_auc_score(true_labels, predicted_probabilities)
    else:
        return "ROC AUC score is not defined for a single class."

import torch
import math
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

def print_summary_images(images, labels, preds, stage, attack_name, dataset_name,
                         num_clean_elements, num_adv_elements, num_combined_elements,
                         num_epoch, num_test_images, label_map=None, defense_type="adversarial_training"):

    # Auto label_map switch
    if label_map is None:
        if dataset_name == "TrafficSigns":
            label_map = {
                0: "Speed Limit",
                1: "Stop Sign",
                2: "Traffic Light",
                3: "Yield"
            }
        elif dataset_name == "TinyImageNet":
            print("TinyImageNet label_map missing – fallback to index labels.")
            label_map = {i: f"Class_{i}" for i in range(200)}
        elif dataset_name == "MNIST":
            label_map = {i: str(i) for i in range(10)}
        elif dataset_name == "EMNIST":
            label_map = {i: str(i) for i in range(10)}  # EMNIST digits: 0-9
        elif dataset_name == "USPS":
            label_map = {i: str(i) for i in range(10)}
        elif dataset_name == "ReflectSign":
            print("ReflectSign label_map missing – fallback to index labels.")
            label_map = {i: f"Sign_{i}" for i in range(55)}  # Fallback if not passed
        else:
            label_map = {i: f"Class_{i}" for i in range(100)}


    # No misclassified images
    if len(images) == 0:
        print(f"\n---------------------------------------------------------")
        print(f"No misclassified images for stage: {stage}")
        print(f"Attack: {attack_name}")
        print(f"Dataset: {dataset_name}")
        print(f"Training Epochs: {num_epoch}")
        if stage == "Clean":
            print(f"Trained Clean Images: {num_clean_elements}")
            accuracy = test_acc
        elif stage == "No Defense Attack":
            # Don't show training images here - this is the ATTACK stage
            print(f"Model: Clean Model (No Defense)")
            accuracy = clean_attack_adv_test_accuracy
        elif stage == "w/ Defense Attack":
            # Show what defense was used
            if defense_type == "adversarial_training":
                print(f"Model: Adversarially Trained (Clean + Adversarial Training)")
                print(f"Training Images Used: {num_combined_elements}")
            elif defense_type == "randomization":
                print(f"Model: Clean Model + Randomization Defense at Test Time")
                print(f"Training Images Used: {num_clean_elements} (no retraining)")
            elif defense_type == "input_transformation":
                print(f"Model: Retrained on Transformed Images")
                print(f"Training Images Used: {num_combined_elements}")
            accuracy = retrained_accuracy
        print(f"Test Images: {num_test_images}")
        print(f"Accuracy: {accuracy:.2f}")

        true_positives = int(accuracy * num_test_images)
        false_positives = 0
        false_negatives = num_test_images - true_positives

        precision_value = precision(true_positives, false_positives)
        recall_value = recall(true_positives, false_negatives)
        f1_score_value = f1_score(precision_value, recall_value)

        y_true = [1]*true_positives + [0]*false_negatives
        y_score = [1]*true_positives + [0]*false_negatives

        if len(y_true) >= 2 and len(set(y_true)) > 1:
            try:
                auc = roc_auc_score(y_true, y_score)
            except Exception as e:
                auc = f"ROC AUC Score error: {str(e)}"
        else:
            auc = "ROC AUC Score: N/A (not enough positive/negative samples)"

        print(f"Precision: {precision_value:.2f}")
        print(f"Recall: {recall_value:.2f}")
        print(f"F1-score: {f1_score_value:.2f}")
        print(f"ROC AUC Score: {auc:.2f}" if isinstance(auc, float) else auc)
        print(f"---------------------------------------------------------\n")
        return

    # Misclassified image summary
    print(f"\n---------------------------------------------------------")
    print(f"Number of misclassified images for {stage}: {len(images)}")
    print(f"Attack: {attack_name}")
    print(f"Dataset: {dataset_name}")
    print(f"Training Epochs: {num_epoch}")
    if stage == "Clean":
        print(f"Trained Clean Images: {num_clean_elements}")
        accuracy = test_acc
    elif stage == "No Defense Attack":
        # Don't show training images here - this is the ATTACK stage, not training
        # Just show it's testing the clean model under attack
        print(f"Model: Clean Model (No Defense)")
        accuracy = clean_attack_adv_test_accuracy
    elif stage == "w/ Defense Attack":
        # Show what defense was used
        if defense_type == "adversarial_training":
            print(f"Model: Adversarially Trained (Clean + Adversarial Training)")
            print(f"Training Images Used: {num_combined_elements}")
        elif defense_type == "randomization":
            print(f"Model: Clean Model + Randomization Defense at Test Time")
            print(f"Training Images Used: {num_clean_elements} (no retraining)")
        elif defense_type == "input_transformation":
            print(f"Model: Retrained on Transformed Images")
            print(f"Training Images Used: {num_combined_elements}")
        accuracy = retrained_accuracy
    print(f"Test Images: {num_test_images}")
    print(f"Accuracy: {accuracy:.2f}")

    # Metrics
    true_positives = int(accuracy * num_test_images)
    false_positives = len(images)
    false_negatives = max(0, num_test_images - true_positives - false_positives)

    precision_value = precision(true_positives, false_positives)
    recall_value = recall(true_positives, false_negatives)
    f1_score_value = f1_score(precision_value, recall_value)

    y_true = [1]*true_positives + [0]*false_negatives
    y_scores = [1]*true_positives + [0]*false_negatives

    if len(y_true) > 0 and len(set(y_true)) > 1:
        try:
            auc = roc_auc_score(y_true, y_scores)
        except Exception as e:
            auc = f"ROC AUC Score error: {str(e)}"
    else:
        auc = "N/A (not enough class diversity)"

    print(f"Precision: {precision_value:.2f}")
    print(f"Recall: {recall_value:.2f}")
    print(f"F1-score: {f1_score_value:.2f}")
    print(f"ROC AUC Score: {auc:.2f}" if isinstance(auc, float) else auc)
    print(f"---------------------------------------------------------\n")

    # Plot misclassified images
    misclassifications = {}
    fig = plt.figure(figsize=(6, len(images)))
    rows = math.ceil(len(images) / 2)
    cols = 2 if len(images) > 16 else min(len(images), 2)

    for i, (img, label, pred) in enumerate(zip(images, labels, preds)):
        ax = fig.add_subplot(rows, cols, i + 1)
        if img.ndim == 2:
            img = img.unsqueeze(0)
        img = img.permute(1, 2, 0)

        label_str = label_map.get(label.item(), str(label.item())) if isinstance(label, torch.Tensor) else label_map.get(label, str(label))
        pred_str = label_map.get(pred.item(), str(pred.item())) if isinstance(pred, torch.Tensor) else label_map.get(pred, str(pred))

        key = f"{label_str} -> {pred_str}"
        misclassifications[key] = misclassifications.get(key, 0) + 1

        img_disp = img.squeeze()
        if img_disp.dtype == torch.float32 or img_disp.dtype == torch.float64:
            img_disp = (img_disp + 1.0) / 2.0
        ax.imshow(img_disp, cmap='gray' if img.shape[-1] == 1 else None)
        ax.set_title(f"{label_str} → {pred_str}", fontsize=9)
        ax.axis("off")

    print("Misclassifications:")
    for k, v in misclassifications.items():
        print(f"{k}: {v}")

    plt.tight_layout()
    plt.show()

print("Number of clean images in training set:", num_clean_images)
print(f"Number of clean batches: {num_clean_batches}")

print(f"Number of adversarial images: {num_adv_images}")
print(f"Number of adversarial batches: {num_adv_batches}")

print(f"Number of combined images: {num_combined_images}")
print(f"Number of combined batches: {num_combined_batches}")

import tabulate

def summarize_performance(clean_loss, clean_acc, adv_loss, adv_acc, retrain_loss, retrain_acc,
                          clean_misclassified_images, adv_misclassified_images, retrained_misclassified_images,
                          clean_misclassified_labels, adv_misclassified_labels, retrained_misclassified_labels,
                          clean_misclassified_preds,adv_misclassified_preds, retrained_misclassified_preds, defense_type, randomization_defense, input_transformation, dataset_name=None, label_map=None):

    if (defense_type == "adversarial_training"):
      print("Defense Implemented: Adversarial Training\n")
    elif (defense_type == "randomization"):
      print("Defense Implemented: Randomization\n")
      if (randomization_defense == "random_resizing"):
        print("Randomization Approach: Random Resizing\n")
      elif(randomization_defense == "random_cropping"):
        print("Randomization Approach: Random Cropping\n")
      elif(randomization_defense == "random_rotation"):
        print("Randomization Approach: Random Rotation\n")
      elif(randomization_defense == "combined_randomization"):
        print("Randomization Approach: Combined Randomization\n")
    elif (defense_type == "input_transformation"):
      print("Defense Implemented: Input Transformation\n")
      if (input_transformation == "image_quilting"):
         print("Input Transformation Approach: Image Quilting\n")
      elif (input_transformation == "jpeg_compression"):
        print("Input Transformation Approach: JPEG Compression\n")
      elif (input_transformation == "bit_depth_reduction"):
        print("Input Transformation Approach: Bit Depth Reduction\n")
      elif (input_transformation == "gaussian_noise_defense"):
        print("Input Transformation Approach: Gaussian Noise Defense\n")
      elif (input_transformation == "combined_input_transformation"):
        print("Input Transformation Approach: Combined Input Transformation\n")

    table_data = [["Metric", "Clean", "No Defense Attack", "w/ Defense Attack"],
                  ["Loss", f'{clean_loss:.3g}', f'{adv_loss:.3g}', f'{retrain_loss:.3g}'],
                  ["Accuracy", f'{clean_acc:.0%}', f'{adv_acc:.0%}', f'{retrain_acc:.0%}']]

    print(tabulate.tabulate(table_data, headers="firstrow", tablefmt="grid"))

    print("\nExample Misclassifications:")

    # Access global variable for TinyImageNet only
    if dataset_name == 'TinyImageNet':
        global filtered_test_set
        num_test_images = len(filtered_test_set)
    else:
        num_test_images = len(filtered_test_loader.dataset)  # Number of images, not batches!

    if dataset_name in ["ReflectSign", "TrafficSigns"]:
      print_summary_images(clean_misclassified_images, clean_misclassified_labels, clean_misclassified_preds, "Clean", compounded_attack_name, dataset_name, num_clean_elements, num_adv_elements, num_combined_elements, num_epochs, num_test_images, label_map=idx_to_class, defense_type=defense_type)
      print_summary_images(adv_misclassified_images, adv_misclassified_labels, adv_misclassified_preds, "No Defense Attack",  compounded_attack_name, dataset_name, num_clean_elements, num_adv_elements, num_combined_elements, num_epochs, num_test_images, label_map=idx_to_class, defense_type=defense_type)
      print_summary_images(retrained_misclassified_images, retrained_misclassified_labels, retrained_misclassified_preds, "w/ Defense Attack",  compounded_attack_name, dataset_name, num_clean_elements, num_adv_elements, num_combined_elements, num_epochs, num_test_images, label_map=idx_to_class, defense_type=defense_type)

    elif dataset_name in ["TinyImageNet"]:
      print_summary_images(clean_misclassified_images, clean_misclassified_labels, clean_misclassified_preds, "Clean", compounded_attack_name, dataset_name, num_clean_elements, num_adv_elements, num_combined_elements, num_epochs, num_test_images, label_map=idx_to_class, defense_type=defense_type)
      print_summary_images(adv_misclassified_images, adv_misclassified_labels, adv_misclassified_preds, "No Defense Attack",  compounded_attack_name, dataset_name, num_clean_elements, num_adv_elements, num_combined_elements, num_epochs, num_test_images, label_map=idx_to_class, defense_type=defense_type)
      print_summary_images(retrained_misclassified_images, retrained_misclassified_labels, retrained_misclassified_preds, "w/ Defense Attack",  compounded_attack_name, dataset_name, num_clean_elements, num_adv_elements, num_combined_elements, num_epochs, num_test_images, label_map=idx_to_class, defense_type=defense_type)

    else:
      print_summary_images(clean_misclassified_images, clean_misclassified_labels, clean_misclassified_preds, "Clean", compounded_attack_name, dataset_name, num_clean_elements, num_adv_elements, num_combined_elements, num_epochs, num_test_images, defense_type=defense_type)
      print_summary_images(adv_misclassified_images, adv_misclassified_labels, adv_misclassified_preds, "No Defense Attack",  compounded_attack_name, dataset_name, num_clean_elements, num_adv_elements, num_combined_elements, num_epochs, num_test_images, defense_type=defense_type)
      print_summary_images(retrained_misclassified_images, retrained_misclassified_labels, retrained_misclassified_preds, "w/ Defense Attack",  compounded_attack_name, dataset_name, num_clean_elements, num_adv_elements, num_combined_elements, num_epochs, num_test_images, defense_type=defense_type)

# ============================================================================
# FINAL CALCULATION VERIFICATION
# ============================================================================
print(f"\n{'='*100}")
print("FINAL CALCULATION VERIFICATION")
print(f"{'='*100}\n")

# Verify all accuracy calculations
print("CLEAN MODEL ON CLEAN DATA:")
print(f"  Accuracy: {pre_attack_test_acc:.4f} ({pre_attack_test_acc*100:.2f}%)")
print(f"  Loss: {pre_attack_test_loss:.4f}")
print(f"  Correct: {int(pre_attack_test_acc * len(filtered_test_loader.dataset))} / {len(filtered_test_loader.dataset)}")

print("\nCLEAN MODEL ON ADVERSARIAL DATA (No Defense):")
print(f"  Accuracy: {clean_attack_adv_test_accuracy:.4f} ({clean_attack_adv_test_accuracy*100:.2f}%)")
print(f"  Loss: {clean_attack_test_loss:.4f}")
print(f"  Correct: {int(clean_attack_adv_test_accuracy * len(attacked_test_loader.dataset))} / {len(attacked_test_loader.dataset)}")

print("\nDEFENSE MODEL ON ADVERSARIAL DATA (With Defense):")
print(f"  Accuracy: {post_attack_test_acc:.4f} ({post_attack_test_acc*100:.2f}%)")
print(f"  Loss: {retrained_test_loss:.4f}")
print(f"  Correct: {int(post_attack_test_acc * len(attacked_test_loader.dataset))} / {len(attacked_test_loader.dataset)}")

# ============================================================================
# ROBUSTNESS METRICS
# ============================================================================
print(f"\n{'='*100}")
print("ROBUSTNESS METRICS")
print(f"{'='*100}\n")

# 1. Attack Success Rate (how much did attack degrade performance)
attack_degradation = pre_attack_test_acc - clean_attack_adv_test_accuracy
attack_success_rate = (attack_degradation / pre_attack_test_acc) * 100 if pre_attack_test_acc > 0 else 0

print(f"ATTACK EFFECTIVENESS:")
print(f"  Clean Accuracy: {pre_attack_test_acc*100:.2f}%")
print(f"  Attacked (No Defense): {clean_attack_adv_test_accuracy*100:.2f}%")
print(f"  Accuracy Drop: {attack_degradation*100:.2f} percentage points")
print(f"  Attack Success Rate: {attack_success_rate:.2f}%")

# 2. Defense Recovery (how much did defense recover from attack)
defense_recovery = post_attack_test_acc - clean_attack_adv_test_accuracy
defense_recovery_percent = (defense_recovery / attack_degradation) * 100 if attack_degradation > 0 else 0

print(f"\nDEFENSE EFFECTIVENESS:")
print(f"  Attacked (No Defense): {clean_attack_adv_test_accuracy*100:.2f}%")
print(f"  With Defense: {post_attack_test_acc*100:.2f}%")
print(f"  Accuracy Recovery: {defense_recovery*100:.2f} percentage points")
print(f"  Recovery Rate: {defense_recovery_percent:.2f}% of lost accuracy")

# 3. Overall Robustness
robustness_improvement_absolute = post_attack_test_acc - clean_attack_adv_test_accuracy

# Fix for "inf%" when both are 0%
if clean_attack_adv_test_accuracy > 0:
    robustness_improvement_relative = (robustness_improvement_absolute / clean_attack_adv_test_accuracy) * 100
elif robustness_improvement_absolute > 0:
    # Defense recovered something from 0% (actual improvement from nothing)
    robustness_improvement_relative = float('inf')
else:
    # Both 0%, no improvement at all
    robustness_improvement_relative = 0.0

remaining_gap = pre_attack_test_acc - post_attack_test_acc

print(f"\nOVERALL ROBUSTNESS:")
print(f"  Absolute Improvement: {robustness_improvement_absolute*100:.2f} percentage points")
if robustness_improvement_relative == float('inf'):
    print(f"  Relative Improvement: Infinite (recovered from 0%)")
elif robustness_improvement_relative == 0.0 and robustness_improvement_absolute == 0.0:
    print(f"  Relative Improvement: 0.00% (no improvement)")
else:
    print(f"  Relative Improvement: {robustness_improvement_relative:.2f}% increase over no defense")
print(f"  Remaining Gap to Clean: {remaining_gap*100:.2f} percentage points")
print(f"  Defense Robustness Score: {post_attack_test_acc*100:.2f}%")

# 4. Defense Classification
if post_attack_test_acc > 0.8:
    defense_rating = "EXCELLENT - Strong defense against attack"
elif post_attack_test_acc > 0.6:
    defense_rating = "GOOD - Moderate defense effectiveness"
elif post_attack_test_acc > 0.4:
    defense_rating = "FAIR - Partial defense effectiveness"
elif post_attack_test_acc > 0.2:
    defense_rating = "WEAK - Limited defense effectiveness"
else:
    defense_rating = "POOR - Minimal defense effectiveness"

print(f"\nDEFENSE RATING: {defense_rating}")

print(f"\n{'='*100}\n")

summarize_performance(pre_attack_test_loss, pre_attack_test_acc, clean_attack_test_loss, clean_attack_adv_test_accuracy, retrained_test_loss, post_attack_test_acc,
                          pre_attack_misclassified_images, adv_misclassified_images, retrained_misclassified_images,
                          pre_attack_misclassified_labels, adv_misclassified_labels, retrained_misclassified_labels,
                          pre_attack_misclassified_preds,adv_misclassified_preds, retrained_misclassified_preds, defense_type, randomization_defense, input_transformation, dataset_name=dataset_name,label_map=idx_to_class)

!pip install -q fpdf
from google.colab import drive
import os
from contextlib import redirect_stdout
from io import StringIO
from fpdf import FPDF
from datetime import datetime

def save_pdf_to_gdrive(
    defense_type,
    randomization_defense,
    input_transformation,
    compounded_attack_name,
    dataset_name,
    test_loss, test_acc,
    clean_attack_test_loss, clean_attack_adv_test_accuracy,
    retrained_test_loss, retrained_accuracy,
    clean_misclassified_images, adv_misclassified_images, retrained_misclassified_images,
    clean_misclassified_labels, adv_misclassified_labels, retrained_misclassified_labels,
    clean_misclassified_preds, adv_misclassified_preds, retrained_misclassified_preds,
    num_clean_elements, num_adv_elements, num_combined_elements,
    num_epochs, filtered_test_loader,
    label_map=None
):
    from datetime import datetime
    from contextlib import redirect_stdout
    from io import StringIO
    from google.colab import drive
    import os

    # Mount drive
    drive.mount('/content/drive', force_remount=True)

    # Short codes - use proper abbreviations
    rd_map = {
        "random_resizing": "resizing",
        "random_cropping": "cropping",
        "random_rotation": "rotation",
        "combined_randomization": "combined"
    }
    rd = rd_map.get(randomization_defense, randomization_defense[:3])
    it_map = {
        "image_quilting": "quilting",
        "jpeg_compression": "jpeg",
        "bit_depth_reduction": "bitdepth",
        "gaussian_noise_defense": "gaussian",
        "combined_input_transformation": "combined"
    }
    it = it_map.get(input_transformation, input_transformation[:3])
    dn = dataset_name  # Use full dataset name, not abbreviation

    # Timestamp
    now = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Build filename based on defense type - only include relevant info
    if defense_type == "adversarial_training":
        file_name = f"{defense_type}_{compounded_attack_name}_{dn}_{now}.txt"
    elif defense_type == "randomization":
        file_name = f"{defense_type}_{compounded_attack_name}_{rd}_{dn}_{now}.txt"
    elif defense_type == "input_transformation":
        file_name = f"{defense_type}_{compounded_attack_name}_{it}_{dn}_{now}.txt"
    else:
        # Fallback to old format
        file_name = f"{defense_type}_{compounded_attack_name}_{rd}_{it}_{dn}_{now}.txt"

    save_path = f"/content/drive/My Drive/results/{file_name}"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Calculate robustness metrics
    attack_degradation = pre_attack_test_acc - clean_attack_adv_test_accuracy
    attack_success_rate = (attack_degradation / pre_attack_test_acc) * 100 if pre_attack_test_acc > 0 else 0
    defense_recovery = post_attack_test_acc - clean_attack_adv_test_accuracy
    defense_recovery_percent = (defense_recovery / attack_degradation) * 100 if attack_degradation > 0 else 0
    robustness_improvement_absolute = post_attack_test_acc - clean_attack_adv_test_accuracy

    # Fix for "inf%" when both are 0%
    if clean_attack_adv_test_accuracy > 0:
        robustness_improvement_relative = (robustness_improvement_absolute / clean_attack_adv_test_accuracy) * 100
    elif robustness_improvement_absolute > 0:
        robustness_improvement_relative = float('inf')
    else:
        robustness_improvement_relative = 0.0

    remaining_gap = pre_attack_test_acc - post_attack_test_acc

    # Capture output
    output_buffer = StringIO()
    with redirect_stdout(output_buffer):
        summarize_performance(
            pre_attack_test_loss, pre_attack_test_acc,
            clean_attack_test_loss, clean_attack_adv_test_accuracy,
            retrained_test_loss, post_attack_test_acc,
            pre_attack_misclassified_images, adv_misclassified_images, retrained_misclassified_images,
            pre_attack_misclassified_labels, adv_misclassified_labels, retrained_misclassified_labels,
            pre_attack_misclassified_preds, adv_misclassified_preds, retrained_misclassified_preds,
            defense_type, randomization_defense, input_transformation,
            dataset_name=dataset_name,
            label_map=None
        )

    # Save output to Drive with robustness metrics
    with open(save_path, "w") as f:
        f.write(output_buffer.getvalue())
        f.write("\n" + "="*100 + "\n")
        f.write("ROBUSTNESS METRICS\n")
        f.write("="*100 + "\n\n")
        f.write(f"ATTACK EFFECTIVENESS:\n")
        f.write(f"  Clean Accuracy: {pre_attack_test_acc*100:.2f}%\n")
        f.write(f"  Attacked (No Defense): {clean_attack_adv_test_accuracy*100:.2f}%\n")
        f.write(f"  Accuracy Drop: {attack_degradation*100:.2f} percentage points\n")
        f.write(f"  Attack Success Rate: {attack_success_rate:.2f}%\n\n")
        f.write(f"DEFENSE EFFECTIVENESS:\n")
        f.write(f"  Attacked (No Defense): {clean_attack_adv_test_accuracy*100:.2f}%\n")
        f.write(f"  With Defense: {post_attack_test_acc*100:.2f}%\n")
        f.write(f"  Accuracy Recovery: {defense_recovery*100:.2f} percentage points\n")
        f.write(f"  Recovery Rate: {defense_recovery_percent:.2f}% of lost accuracy\n\n")
        f.write(f"OVERALL ROBUSTNESS:\n")
        f.write(f"  Absolute Improvement: {robustness_improvement_absolute*100:.2f} percentage points\n")
        if robustness_improvement_relative == float('inf'):
            f.write(f"  Relative Improvement: Infinite (recovered from 0%)\n")
        elif robustness_improvement_relative == 0.0 and robustness_improvement_absolute == 0.0:
            f.write(f"  Relative Improvement: 0.00% (no improvement)\n")
        else:
            f.write(f"  Relative Improvement: {robustness_improvement_relative:.2f}% increase over no defense\n")
        f.write(f"  Remaining Gap to Clean: {remaining_gap*100:.2f} percentage points\n")
        f.write(f"  Defense Robustness Score: {post_attack_test_acc*100:.2f}%\n")

    return save_path

pdf_path = save_pdf_to_gdrive(
    defense_type,
    randomization_defense,
    input_transformation,
    compounded_attack_name,
    dataset_name,
    pre_attack_test_loss, pre_attack_test_acc,
    clean_attack_test_loss, clean_attack_adv_test_accuracy,
    retrained_test_loss, post_attack_test_acc,
    pre_attack_misclassified_images, adv_misclassified_images, retrained_misclassified_images,
    pre_attack_misclassified_labels, adv_misclassified_labels, retrained_misclassified_labels,
    pre_attack_misclassified_preds, adv_misclassified_preds, retrained_misclassified_preds,
    num_clean_elements,
    num_adv_elements,
    num_combined_elements,
    num_epochs,
    filtered_test_loader,
    label_map=None
)

print(pdf_path)